# Backbone experiments for zero-shot anomaly detection (pptx-matched)

**What this notebook reports.** The **Backbone Benchmarking** track from
*Contrastive VLMs in Anomaly Detection* (pptx slide 23), run under the
**Zero-shot benchmarking** protocol (slide 21), with natural corruptions from
slides 18/19.

**Backbones (slide 23 + CLIP baseline):** TIPS 1 (`tips`), TIPS 2 (`tips_v2`),
**CLIP**, **SigLIP2**, **DINOv2.txt**. DINOv3.txt is listed on the slide but has
no public language-aligned weights yet (`model_configs/dinov3.txt`).

**The setting is AnomalyCLIP's, minus the internal adaptation.** AnomalyCLIP
(Zhou et al., ICLR 2024) combines object-agnostic prompt learning with changes
*inside* the frozen encoder — DPAM attention surgery, learnable visual tokens and
projection layers on the intermediate features. This notebook keeps the
protocol, the losses, the metrics and the object-agnostic prompt design, and
removes everything internal:

> **The only trainable parameters anywhere in this notebook are the text prompt
> context vectors. Every encoder is frozen, no attention layer is modified, no
> adapter or projection head is introduced, and no visual prompt is learned.**

Slide 23 asks to measure **fixed and learnable** prompt learning. The notebook
also scores **decoupled** (fixed image score + learned map) — Tipsomaly's recipe —
under the same prompt-only constraint.

## Protocol (pptx slide 21)

Only the **test** split of either dataset is ever read, and the two roles are
kept strictly disjoint:

| Prompts learned on | Evaluated on | Category overlap |
| --- | --- | --- |
| MVTec test split (15 categories) | VisA test split (12 categories) | none |
| VisA test split (12 categories) | MVTec test split (15 categories) | none |

| Knob | Pptx value |
| --- | --- |
| Random seed | **111** (slide 21); multi-seed supported via `SEEDS` / separate `OUT_ROOT` |
| Corruptions | Slide 18/19 selection, severities **1–3** (slide 21) |
| Artefacts | Store **low-res anomaly maps** and **anomaly scores** |
| Metrics | Pixel AUROC / F1-max / AUPRO / threshold; Image AUROC / F1-max / AP / threshold |
| Aggregation | Category-level and dataset-level |

Defaults: `MODELS = PPTX_MODELS`. For a cheap rehearsal, set
`MODELS = DEMO_MODELS` (`tips`, `clip` only).

**Paper coverage note.** Tipsomaly ([arXiv:2602.03594](https://arxiv.org/abs/2602.03594))
reports TIPS and a SigLIP2 ablation. It does **not** report TIPS-v2 or
DINOv2.txt — never invent published numbers for those.

## Cell 1 — Configuration (pptx-matched defaults)

Every choice that defines the experiment lives in this one cell. Defaults follow
the pptx **Zero-shot** (slide 21) + **Backbone Benchmarking** (slide 23) design.

| Constant | Pptx-matched default |
| --- | --- |
| `MODELS` | `PPTX_MODELS` = tips, tips_v2, clip, siglip2, dinov2 |
| `DEMO_MODELS` | tips, clip — optional cheap rehearsal only |
| `PROMPT_MODES` | fixed, learned, decoupled (pptx requires fixed+learned) |
| `SEED` / `SEEDS` | 111 / (111,) |
| `SEVERITIES` | (1, 2, 3); use `SLIDE19_SEVERITIES` for 4 levels |
| `DATASETS` / `PROTOCOL` | MVTec ↔ VisA, test-only, exclusive |
| Trainable params | prompt context only (`N_CTX=8`) |

Hyperparameters matched to Tipsomaly where the pptx is silent: **518×518** inputs,
Adam **lr 1e-3**, **betas (0.5, 0.999)**, **2** epochs, batch size **8**.

In [ ]:
# =============================================================================
# CELL 1 -- Experiment configuration (pptx-matched defaults)
# =============================================================================
# Source of truth for the experimental design:
#   Contrastive VLMs in Anomaly Detection.pptx
#     slide 21  Zero-shot benchmarking protocol
#     slide 23  Backbone Benchmarking (this project's track)
#     slide 18/19  Natural corruptions
#     slide 10/24  Complexity / calibration viewpoints (reported alongside)
#
# Constraint (AnomalyCLIP setting minus internal adaptation):
#   The only trainable parameters are text prompt context vectors.
#   No DPAM / attention surgery, no visual prompts, no adapters, no proj heads.
import os
import sys

SEED = 111
# Slide 10 asks for multi-seed reproducibility. Primary protocol seed is 111
# (slide 21). Add more seeds here to repeat the full sweep, e.g. (111, 222, 333).
SEEDS = (111,)

# --- Backbones (pptx slide 23) ----------------------------------------------
# Slide 23 names: SigLIP2, TIPS 1 / TIPS 2, DINOv2.txt / DINOv3.txt.
# CLIP is included as the baseline whose limitations those models claim to fix
# (slide 11 / slide 23 preamble). DINOv3.txt has no public VLM weights yet.
TIPS_VERSION = "v1"          # used by the "tips" (TIPS 1) factory
TIPS_VARIANT = "L"           # "S" | "B" | "L" | "So400m" | "g"
CLIP_BACKBONE = "ViT-L/14@336px"
SIGLIP2_MODEL = "hf-hub:timm/ViT-L-16-SigLIP2-384"

INPUT_SIZE = 518

# Full pptx backbone track (+ CLIP baseline). DINOv3 is intentionally absent.
PPTX_MODELS = ("tips", "tips_v2", "clip", "siglip2", "dinov2")
# Minimal GPU rehearsal / Colab smoke (not the pptx-complete set).
DEMO_MODELS = ("tips", "clip")
# Default = pptx-complete experimental design.
MODELS = PPTX_MODELS
AVAILABLE_MODELS = PPTX_MODELS + ("dinov3",)  # dinov3 blocked — see model_configs/

DENSE_LAYER_FRACTIONS = {
    "tips": (1.0,),
    "tips_v2": (1.0,),
    "clip": (0.25, 0.5, 0.75, 1.0),
    "siglip2": (1.0,),
    "dinov2": (1.0,),
}
SHARED_DENSE_LAYERS = None

# MaskCLIP values trick on FROZEN weights (not a learnable adaptation).
USE_VALUE_ATTENTION = True

# --- Prompting (pptx slide 23: fixed + learnable) ---------------------------
# pptx requires measuring fixed and learnable prompt learning. `decoupled`
# (fixed score + learned map) is included as the Tipsomaly recipe under the
# same prompt-only constraint; it does not add internal encoder parameters.
PROMPT_MODES = ("fixed", "learned", "decoupled")
PPTX_REQUIRED_PROMPT_MODES = ("fixed", "learned")

N_CTX = 8
LEARNABLE_SUFFIX = {"normal": "object", "anomalous": "damaged object"}
FIXED_PROMPT_CLASS_NAME = None

LOSS_MODE = "local"
FOCAL_GAMMA = 2.0
IMAGE_LOSS_WEIGHT = 1.0
PIXEL_LOSS_WEIGHT = 1.0

GLOBAL_TOKEN = "spatial"
ADD_LOCAL_EVIDENCE = True

# --- Optimisation ------------------------------------------------------------
EPOCHS = 2
BATCH_SIZE = 8
LR = 1e-3
ADAM_BETAS = (0.5, 0.999)
WEIGHT_DECAY = 0.0
GRAD_CLIP = 1.0

# --- Where files live --------------------------------------------------------
# Colab, Kaggle and a laptop each put writable scratch somewhere different, so
# the roots are derived instead of hardcoded. Override with TIPS_WORKSPACE, or
# just reassign DATA_ROOT / OUT_ROOT below.
def _detect_platform():
    # Prefer explicit markers over bare path checks: on Windows a leftover
    # C:\content directory must NOT be treated as Google Colab.
    if os.path.isdir("/kaggle/working") and (
            os.path.isdir("/kaggle/input") or os.environ.get("KAGGLE_KERNEL_RUN_TYPE")):
        return "kaggle", "/kaggle/working"
    try:
        import google.colab  # noqa: F401
        return "colab", "/content"
    except ImportError:
        pass
    if os.path.isdir("/content") and not sys.platform.startswith("win"):
        return "colab", "/content"
    return "local", os.getcwd()


PLATFORM, _default_workspace = _detect_platform()
WORKSPACE_ROOT = os.environ.get("TIPS_WORKSPACE", _default_workspace)

# --- Datasets (pptx slide 21) -----------------------------------------------
# ONLY test sets, used for training and evaluation exclusively and disjointly.
# Override with env TIPS_DATA_ROOT, or reassign DATA_ROOT below. On a tight
# system drive, point this at a larger disk (MVTec+VisA need ~15+ GB free).
_default_data = os.path.join(WORKSPACE_ROOT, "data")
# If a previous run already placed archives on a larger Windows drive, reuse it.
if (PLATFORM == "local" and os.path.isdir(r"E:\tips_clip_ad_data")
        and not os.environ.get("TIPS_DATA_ROOT")):
    _default_data = r"E:\tips_clip_ad_data"
DATA_ROOT = os.environ.get("TIPS_DATA_ROOT", _default_data)
# Prefer VisA / MVTec trees already next to the notebook (common local unpack).
# Official VisA extracts as VisA_20220922/; MVTec as mvtec_anomaly_detection/.
# Keep those names — no rename required. DATA_ROOT/<name> remains the download
# default only when nothing is found on disk yet (PROJECT_ROOT is finalized in
# the setup cell; cwd / WORKSPACE_ROOT cover the notebook path here).
_VISA_DIR_NAMES = ("VisA_20220922", "visa_20220922", "VisA")
_MVTEC_DIR_NAMES = ("mvtec_anomaly_detection", "MVTec_AD", "mvtec")
_dataset_project_bases = (
    os.getcwd(),
    WORKSPACE_ROOT,
    os.path.join(WORKSPACE_ROOT, "tips_clip_ad"),
    os.path.join(os.getcwd(), "tips_clip_ad"),
)
_dataset_data_bases = (DATA_ROOT,)
VISA_ROOT = os.path.join(DATA_ROOT, "VisA")  # download/legacy default
# Prefer project-local MVTec next to the notebook (same place as VisA unpacks).
# DATA_ROOT is only the fallback when no project-local VisA/MVTec signal exists.
MVTEC_ROOT = os.path.join(
    next((b for b in _dataset_project_bases if b), DATA_ROOT),
    "mvtec_anomaly_detection",
)
_visa_found = False
for _base in _dataset_project_bases + _dataset_data_bases:
    if not _base:
        continue
    for _name in _VISA_DIR_NAMES:
        _candidate = os.path.join(_base, _name)
        if os.path.isfile(os.path.join(_candidate, "split_csv", "1cls.csv")):
            VISA_ROOT = _candidate
            _visa_found = True
            break
    if _visa_found:
        break
if not _visa_found:
    # Flat extract: split_csv/ lives directly under the project root.
    for _base in _dataset_project_bases:
        if _base and os.path.isfile(os.path.join(_base, "split_csv", "1cls.csv")):
            VISA_ROOT = _base
            _visa_found = True
            break

_mvtec_found = False
# 1) Complete tree (bottle/test) — project-local aliases BEFORE DATA_ROOT / E:\...
for _base in _dataset_project_bases:
    if not _base:
        continue
    for _name in _MVTEC_DIR_NAMES:
        _candidate = os.path.join(_base, _name)
        if os.path.isdir(os.path.join(_candidate, "bottle", "test")):
            MVTEC_ROOT = _candidate
            _mvtec_found = True
            break
    if _mvtec_found:
        break
if not _mvtec_found:
    # 2) Existing (possibly incomplete / still downloading) project-local folder
    for _base in _dataset_project_bases:
        if not _base:
            continue
        for _name in _MVTEC_DIR_NAMES:
            _candidate = os.path.join(_base, _name)
            if os.path.isdir(_candidate):
                MVTEC_ROOT = _candidate
                _mvtec_found = True
                break
        if _mvtec_found:
            break
if not _mvtec_found:
    # 3) VisA already under a project base → default MVTec alongside it
    #    (do not fall back to E:\... when the intended unpack is next to VisA).
    _visa_norm = os.path.normcase(os.path.normpath(VISA_ROOT))
    for _base in _dataset_project_bases:
        if not _base:
            continue
        _base_norm = os.path.normcase(os.path.normpath(_base))
        if _visa_norm == _base_norm or _visa_norm.startswith(_base_norm + os.sep):
            MVTEC_ROOT = os.path.join(_base, "mvtec_anomaly_detection")
            _mvtec_found = True
            break
if not _mvtec_found:
    # 4) Legacy / download default under DATA_ROOT (complete tree if present)
    for _base in _dataset_data_bases:
        if not _base:
            continue
        for _name in _MVTEC_DIR_NAMES:
            _candidate = os.path.join(_base, _name)
            if os.path.isdir(os.path.join(_candidate, "bottle", "test")):
                MVTEC_ROOT = _candidate
                _mvtec_found = True
                break
        if _mvtec_found:
            break
    if not _mvtec_found:
        MVTEC_ROOT = os.path.join(DATA_ROOT, "mvtec_anomaly_detection")
del (_VISA_DIR_NAMES, _MVTEC_DIR_NAMES, _dataset_project_bases, _dataset_data_bases,
     _visa_found, _mvtec_found)
try:
    del _base, _name, _candidate, _visa_norm, _base_norm
except NameError:
    pass
DATASETS = ("mvtec", "visa")

PROTOCOL = (
    ("mvtec", "visa"),
    ("visa", "mvtec"),
)

# --- Corruptions (pptx slides 18/19 + 21) -----------------------------------
# Slide 19 taxonomy (4 groups). Slide 21: apply selected corruptions at levels
# 1–3. Slide 19 also mentions 4 severity levels; magnitudes for severity 4 are
# defined below — set SEVERITIES = SLIDE19_SEVERITIES to use all four.
CORRUPTION_GROUPS = {
    "noise": ("gaussian_noise", "shot_noise", "impulse_noise"),
    "blur": ("defocus_blur", "motion_blur", "zoom_blur"),
    "photometric": ("brightness", "contrast"),
    "geometric": ("rotation", "zoom_scale", "shift"),
}
SEVERITIES = (1, 2, 3)                 # slide 21 (ZSAD / backbone protocol)
SLIDE19_SEVERITIES = (1, 2, 3, 4)      # slide 19 "4 severity levels"
INCLUDE_CLEAN = True
CORRUPT_TRAINING = False               # prompts learned on clean only

GEOMETRIC_MAGNITUDES = {
    "rotation": (5.0, 10.0, 15.0, 20.0),
    "zoom_scale": (1.05, 1.10, 1.15, 1.20),
    "shift": (0.02, 0.04, 0.06, 0.08),
}

# --- Artefacts & metrics (pptx slide 21) ------------------------------------
MAP_RES = 64
PAPER_COMPARISON_RES = 256

OUT_ROOT = os.path.join(WORKSPACE_ROOT, "results")
ARTIFACT_DIR = os.path.join(OUT_ROOT, "artifacts")
CHECKPOINT_DIR = os.path.join(OUT_ROOT, "prompts")
TABLE_DIR = os.path.join(OUT_ROOT, "tables")
COMPLEXITY_DIR = os.path.join(OUT_ROOT, "complexity")
RESUME = True

GAUSSIAN_SIGMA = 1.0
TOPK_FRACTION = 0.0

# --- Runtime ----------------------------------------------------------------
# Windows spawns worker processes, which cannot pickle a Dataset class defined
# in a notebook cell, so loading stays in-process there.
NUM_WORKERS = 0 if sys.platform.startswith("win") else 4
AMP = True
MAX_TRAIN_IMAGES_PER_CATEGORY = None

# --- Pptx protocol self-check (printed once when the cell runs) -------------
assert SEED == 111 or SEED in SEEDS, "slide 21 primary seed is 111"
assert set(PPTX_REQUIRED_PROMPT_MODES) <= set(PROMPT_MODES), \
    "pptx slide 23 requires fixed and learnable prompt modes"
assert set(SEVERITIES) <= {1, 2, 3, 4, 5}, "ImageNet-C severities are 1..5"
assert not CORRUPT_TRAINING, "pptx robustness: train clean, evaluate corrupted"
assert DATASETS == ("mvtec", "visa"), "slide 21 main datasets are MVTec and VisA"

print("PPTX protocol defaults:")
print(f"  MODELS={MODELS}  (DEMO_MODELS={DEMO_MODELS})")
print(f"  PROMPT_MODES={PROMPT_MODES}  (required={PPTX_REQUIRED_PROMPT_MODES})")
print(f"  SEED={SEED}  SEEDS={SEEDS}  SEVERITIES={SEVERITIES}")
print(f"  DATASETS={DATASETS}  PROTOCOL={PROTOCOL}")
print(f"  prompt-learning only; encoders frozen; no internal adapters")
print(f"\nplatform={PLATFORM}  workspace={WORKSPACE_ROOT}")
print(f"  DATA_ROOT ={DATA_ROOT}")
print(f"  MVTEC_ROOT={MVTEC_ROOT}")
print(f"  VISA_ROOT ={VISA_ROOT}")
print(f"  OUT_ROOT  ={OUT_ROOT}")
print(f"  NUM_WORKERS={NUM_WORKERS}")

## Cell 2 — Environment, TIPS source and checkpoints

Installs the dependencies (including OpenCLIP for SigLIP2), clones DeepMind's
TIPS repository, and pre-fetches TIPS checkpoints when a `tips*` model is in
`MODELS`. SigLIP2 / DINOv2.txt weights download on first backbone construction.
`PROJECT_ROOT` / `MODEL_CONFIG_DIR` point at `model_configs/` (needed for
`dinov2.txt`).

Torch must already be installed in the kernel. This cell only installs CLIP /
OpenCLIP / imagecorruptions / sentencepiece and shims `pkg_resources`. Do not
add extra shell-pip cells before it — they often hang with a blank output area
in Jupyter on Windows.

Three details that are easy to get wrong:

- **The import path.** `from tips.pytorch import image_encoder` resolves as an
  implicit namespace package, so the repository must sit at `<root>/tips` with
  `<root>` on `sys.path` — not `<root>/tips/tips`.
- **TensorFlow is not required.** `tips.pytorch.text_encoder` imports
  `tensorflow` and `tensorflow_text` only for its own tokenizer wrapper. This
  notebook tokenises with the `sentencepiece` package instead (same model file,
  same lowercasing, no BOS/EOS token), so the TF imports are stubbed out when
  unavailable and the model definitions stay importable.
- **`imagecorruptions`** is the reference ImageNet-C implementation repackaged
  without the ImageMagick dependency and, unlike the original release, it accepts
  arbitrary image sizes. Severities therefore mean what they mean in every other
  robustness paper.

In [ ]:
# =============================================================================
# CELL 2 -- Environment, TIPS source, and checkpoints
# =============================================================================
import hashlib
import importlib
import json
import os
import subprocess
import sys
import time
import types
import urllib.request
from contextlib import contextmanager

print("setup cell started...", flush=True)

import torch
import torch.nn.functional as F  # noqa: F401  — used by backbone / metric cells
print(f"  torch {torch.__version__} imported", flush=True)

# --- Optional NDJSON tracing -------------------------------------------------
# Inert unless TIPS_DEBUG_LOG points at a file; set it when reporting a problem
# so the failing step can be read back instead of guessed at.
DEBUG_LOG_PATH = os.environ.get("TIPS_DEBUG_LOG")


def debug_log(stage, message, **data):
    if not DEBUG_LOG_PATH:
        return
    try:
        with open(DEBUG_LOG_PATH, "a", encoding="utf-8") as handle:
            handle.write(json.dumps({"stage": stage, "message": message,
                                     "data": data,
                                     "timestamp": int(time.time() * 1000)},
                                    default=str) + "\n")
    except OSError:
        pass


# --- Where the project's own files are ---------------------------------------
ROOT_DIR = WORKSPACE_ROOT
# Prefer a directory that actually contains this project's assets (model_configs/,
# cells/). On Colab the notebook cwd is often /content while the repo lives in a
# subfolder; on a laptop it is the clone root.
_candidates = [os.getcwd(), ROOT_DIR,
               os.path.join(ROOT_DIR, "tips_clip_ad"),
               os.path.dirname(os.getcwd())]
PROJECT_ROOT = next(
    (path for path in _candidates
     if os.path.isdir(os.path.join(path, "model_configs"))),
    os.getcwd(),
)
TIPS_DIR = os.path.join(ROOT_DIR, "tips")
TIPS_CKPT_DIR = os.path.join(ROOT_DIR, "tips_checkpoints")
WEIGHTS_DIR = os.path.join(PROJECT_ROOT, "weights")
MODEL_CONFIG_DIR = os.path.join(PROJECT_ROOT, "model_configs")
os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(TIPS_CKPT_DIR, exist_ok=True)
# Keep torch.hub / HF caches inside the project when possible (resume-friendly).
os.environ.setdefault("TORCH_HOME", os.path.join(WEIGHTS_DIR, "torch"))
os.environ.setdefault("HF_HOME", os.path.join(WEIGHTS_DIR, "huggingface"))
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", os.path.join(WEIGHTS_DIR, "huggingface", "hub"))
os.environ.setdefault("TRANSFORMERS_CACHE", os.path.join(WEIGHTS_DIR, "huggingface", "transformers"))

# Every backbone cell registers itself here. Creating the registry up front
# means one unavailable dependency cannot take the later cells down with it.
BACKBONES = {}
BACKBONE_ERRORS = {}


def register_backbone(name, factory):
    BACKBONES[name] = factory
    BACKBONE_ERRORS.pop(name, None)


def skip_backbone(name, reason):
    """Records why a backbone is unavailable instead of raising mid-notebook."""
    BACKBONE_ERRORS[name] = str(reason)
    print(f"  [skip] backbone {name!r} unavailable: {reason}")


# --- Dependencies helper ------------------------------------------------------
def _pip(*args):
    """Installs quietly and reports failure instead of aborting the notebook."""
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *args],
        capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  [warn] pip install {' '.join(args)} failed:\n"
              f"    {result.stderr.strip().splitlines()[-1] if result.stderr.strip() else '?'}")
    return result.returncode == 0


def _purge_module(name):
    """Drop a failed/partial import so the next importlib call retries cleanly."""
    prefix = name + "."
    for key in list(sys.modules):
        if key == name or key.startswith(prefix):
            del sys.modules[key]


# --- pkg_resources compatibility ---------------------------------------------
# setuptools >= 81 removed pkg_resources, but openai/CLIP does
# `from pkg_resources import packaging` and imagecorruptions also imports it.
# On Python 3.14, pinning an older setuptools often breaks `_distutils_hack`, so
# the durable fix is a shim that exports `packaging` (CLIP's exact import).
def _ensure_pkg_resources():
    try:
        import pkg_resources  # noqa: F401
        return "present"
    except ImportError:
        pass

    if "packaging" not in sys.modules:
        _pip("packaging")
    try:
        import packaging
        import packaging.version  # noqa: F401
    except ImportError as error:
        raise ImportError(
            "Need packaging for the pkg_resources shim (CLIP import). "
            f"pip install packaging failed: {error}"
        ) from error

    shim = types.ModuleType("pkg_resources")

    def resource_filename(package, name):
        module = importlib.import_module(package)
        return os.path.join(os.path.dirname(module.__file__), name)

    shim.resource_filename = resource_filename
    shim.resource_stream = lambda package, name: open(
        resource_filename(package, name), "rb")
    shim.resource_string = lambda package, name: open(
        resource_filename(package, name), "rb").read()
    # CLIP: `from pkg_resources import packaging`
    shim.packaging = packaging

    class DistributionNotFound(Exception):
        pass

    shim.DistributionNotFound = DistributionNotFound
    shim.get_distribution = lambda name: types.SimpleNamespace(version="0")
    # Provide a few other names older libs occasionally touch.
    shim.require = lambda *args, **kwargs: []
    shim.working_set = []
    sys.modules["pkg_resources"] = shim
    return "shimmed"


PKG_RESOURCES_STATUS = _ensure_pkg_resources()
try:
    _setuptools_ver = importlib.import_module("setuptools").__version__
except Exception:  # noqa: BLE001
    _setuptools_ver = "unavailable"
debug_log("setup", "pkg_resources", status=PKG_RESOURCES_STATUS,
          setuptools=_setuptools_ver)


MISSING_DEPENDENCIES = set()

for module, spec in [
    ("clip", "git+https://github.com/openai/CLIP.git"),
    ("imagecorruptions", "imagecorruptions"),
    ("sentencepiece", "sentencepiece"),
    ("skimage", "scikit-image"),
    ("sklearn", "scikit-learn"),
    ("open_clip", "open-clip-torch>=2.31.0"),
]:
    try:
        importlib.import_module(module)
        debug_log("setup", "dependency ok", module=module)
    except ImportError:
        _purge_module(module)
        if _pip(spec):
            importlib.invalidate_caches()
            _ensure_pkg_resources()
            _purge_module(module)
            try:
                importlib.import_module(module)
                debug_log("setup", "dependency ok after install", module=module)
            except ImportError as error:
                MISSING_DEPENDENCIES.add(module)
                debug_log("setup", "dependency still missing",
                          module=module, error=str(error))
        else:
            MISSING_DEPENDENCIES.add(module)

# Prove the critical imports work in *this* kernel (not just that pip ran).
for _probe in ("clip", "imagecorruptions"):
    try:
        _purge_module(_probe)
        importlib.import_module(_probe)
        print(f"  import {_probe}: OK")
        debug_log("setup", "probe import ok", module=_probe)
    except Exception as _probe_error:  # noqa: BLE001
        MISSING_DEPENDENCIES.add(_probe)
        print(f"  import {_probe}: FAILED ({_probe_error})")
        debug_log("setup", "probe import failed", module=_probe,
                  error=str(_probe_error))

# --- TIPS source --------------------------------------------------------------
TIPS_SOURCE_AVAILABLE = os.path.isdir(os.path.join(TIPS_DIR, "pytorch"))
if not TIPS_SOURCE_AVAILABLE:
    try:
        _clone = subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/google-deepmind/tips.git", TIPS_DIR],
            capture_output=True, text=True)
        TIPS_SOURCE_AVAILABLE = os.path.isdir(os.path.join(TIPS_DIR, "pytorch"))
        if not TIPS_SOURCE_AVAILABLE:
            print("  [warn] could not clone google-deepmind/tips:\n"
                  f"    {_clone.stderr.strip().splitlines()[-1] if _clone.stderr.strip() else '?'}")
    except (OSError, subprocess.SubprocessError) as error:
        print(f"  [warn] git unavailable ({error}); TIPS backbones will be skipped")

if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

for name in ("tensorflow", "tensorflow_text"):
    try:
        importlib.import_module(name)
    except ImportError:
        stub = types.ModuleType(name)
        stub.__getattr__ = lambda _attr: None  # inert placeholder
        sys.modules[name] = stub

# --- Checkpoints -------------------------------------------------------------
_V2_FILE_TOKEN = {"B": "b14", "L": "l14", "So400m": "so14", "g": "g14"}
_V1_STEM = {
    "S": "tips_oss_s14_highres_distilled",
    "B": "tips_oss_b14_highres_distilled",
    "L": "tips_oss_l14_highres_distilled",
    "So400m": "tips_oss_so400m14_highres_largetext_distilled",
    "g": "tips_oss_g14_highres",
}
_GCS = "https://storage.googleapis.com/tips_data"
TOKENIZER_URL = _GCS + "/v1_0/checkpoints/tokenizer.model"

# Text tower widths per vision-tower variant; shared between TIPS v1 and v2.
TIPS_TEXT_CONFIG = {
    "S": {"hidden_size": 384, "mlp_dim": 1536, "num_heads": 6, "num_layers": 12},
    "B": {"hidden_size": 768, "mlp_dim": 3072, "num_heads": 12, "num_layers": 12},
    "L": {"hidden_size": 1024, "mlp_dim": 4096, "num_heads": 16, "num_layers": 12},
    "So400m": {"hidden_size": 1152, "mlp_dim": 4304, "num_heads": 16, "num_layers": 27},
    "g": {"hidden_size": 1536, "mlp_dim": 6144, "num_heads": 24, "num_layers": 12},
}


def tips_checkpoint_urls(version, variant):
    if version == "v2":
        if variant not in _V2_FILE_TOKEN:
            raise ValueError(f"TIPSv2 has no {variant} variant")
        stem, base = f"tips_v2_oss_{_V2_FILE_TOKEN[variant]}", f"{_GCS}/v2_0/checkpoints/pytorch"
    elif version == "v1":
        stem, base = _V1_STEM[variant], f"{_GCS}/v1_0/checkpoints/pytorch"
    else:
        raise ValueError(f"unknown TIPS version {version!r}")
    return f"{base}/{stem}_vision.npz", f"{base}/{stem}_text.npz"


def _proxy_is_localhost(proxies=None):
    proxies = proxies if proxies is not None else urllib.request.getproxies()
    return any(
        host in (value or "")
        for value in proxies.values()
        for host in ("127.0.0.1", "localhost", "::1")
    )


def _proxy_failure_text(error):
    reason = getattr(error, "reason", error)
    return f"{error} | {reason}".lower()


def _should_bypass_proxy(error):
    """True when a localhost proxy is dead, refused, or breaking TLS."""
    reason = getattr(error, "reason", error)
    text = _proxy_failure_text(error)
    refused = (
        getattr(reason, "winerror", None) == 10061
        or getattr(reason, "errno", None) in (61, 111)
        or getattr(error, "winerror", None) == 10061
        or "10061" in text
        or "connection refused" in text
        or "actively refused" in text
    )
    tls_or_tunnel = any(token in text for token in (
        "unexpected_eof",
        "eof occurred in violation",
        "ssleof",
        "ssl:",
        "sslerror",
        "wrong version number",
        "proxyerror",
        "tunnel connection failed",
        "cannot connect to proxy",
        "connection reset",
        "broken pipe",
        "timed out",
        "timeout",
    ))
    return refused or tls_or_tunnel


def _direct_opener():
    return urllib.request.build_opener(urllib.request.ProxyHandler({}))


@contextmanager
def without_system_proxy():
    """Disable env + Windows Internet Settings proxies for urllib/requests/HF/torch."""
    keys = ("HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY",
            "http_proxy", "https_proxy", "all_proxy")
    saved_env = {key: os.environ.get(key) for key in keys}
    saved_no = {key: os.environ.get(key) for key in ("NO_PROXY", "no_proxy")}
    real_getproxies = urllib.request.getproxies
    real_registry = getattr(urllib.request, "getproxies_registry", None)
    real_environment = getattr(urllib.request, "getproxies_environment", None)
    try:
        for key in keys:
            os.environ.pop(key, None)
        os.environ["NO_PROXY"] = "*"
        os.environ["no_proxy"] = "*"
        urllib.request.getproxies = lambda: {}  # noqa: E731
        if real_registry is not None:
            urllib.request.getproxies_registry = lambda: {}  # noqa: E731
        if real_environment is not None:
            urllib.request.getproxies_environment = lambda: {}  # noqa: E731
        yield
    finally:
        for key, value in saved_env.items():
            if value is None:
                os.environ.pop(key, None)
            else:
                os.environ[key] = value
        for key, value in saved_no.items():
            if value is None:
                os.environ.pop(key, None)
            else:
                os.environ[key] = value
        urllib.request.getproxies = real_getproxies
        if real_registry is not None:
            urllib.request.getproxies_registry = real_registry
        if real_environment is not None:
            urllib.request.getproxies_environment = real_environment


def _is_forbidden(error):
    text = _proxy_failure_text(error)
    return (
        getattr(error, "code", None) == 403
        or "403" in text
        or "forbidden" in text
    )


def _open_url(request, timeout=120, force_direct=False):
    """Open `request`, preferring the system proxy then falling back to direct.

    Windows 'Internet Settings' often point at 127.0.0.1:10809 (v2rayN / Clash):
      - proxy up   → needed for some hosts (GCS tips_data is 403 without it here)
      - proxy down → WinError 10061 / SSL UNEXPECTED_EOF; then try direct
    Never permanently disable the system proxy: geo-blocked hosts need it.
    """
    if force_direct:
        return _direct_opener().open(request, timeout=timeout)
    try:
        return urllib.request.urlopen(request, timeout=timeout)
    except Exception as error:  # noqa: BLE001
        if not (_should_bypass_proxy(error) and _proxy_is_localhost()):
            raise
        proxies = urllib.request.getproxies()
        print(f"  system proxy {proxies} failed ({error}); "
              "retrying with a direct (no-proxy) connection.", flush=True)
        try:
            return _direct_opener().open(request, timeout=timeout)
        except Exception as direct_error:  # noqa: BLE001
            # Direct 403 usually means the host needs the VPN/proxy; surface the
            # original proxy failure so the user knows to start v2rayN/Clash.
            if _is_forbidden(direct_error):
                print("  direct connection got HTTP 403 — this host likely needs "
                      "the local proxy/VPN. Start v2rayN/Clash on "
                      "127.0.0.1:10809 and retry.", flush=True)
                raise error from direct_error
            raise


def run_with_proxy_fallback(fn, label="download"):
    """Call `fn()`; on proxy/SSL failure retry once with proxies disabled.

    Used for Hugging Face / torch.hub / OpenCLIP loaders that do not go through
    `_download`. GCS TIPS weights should keep using `_download` (proxy-first).
    """
    try:
        return fn()
    except Exception as error:  # noqa: BLE001
        text = _proxy_failure_text(error)
        retryable = _should_bypass_proxy(error) or any(
            token in text for token in (
                "max retries exceeded", "huggingface", "hf hub",
                "connection error", "temporarily unavailable",
            ))
        if not retryable or not _proxy_is_localhost():
            raise
        print(f"  [{label}] retrying without system proxy after: {error}",
              flush=True)
        with without_system_proxy():
            return fn()


def _file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def _remote_content_length(url, force_direct=False):
    """Best-effort Content-Length via GET bytes=0-0 (HEAD is often blocked)."""
    request = urllib.request.Request(
        url,
        headers={
            "User-Agent": "tips-clip-ad/1.0 (research; weight fetch)",
            "Range": "bytes=0-0",
        },
    )
    try:
        with _open_url(request, timeout=60, force_direct=force_direct) as response:
            content_range = response.headers.get("Content-Range") or ""
            if "/" in content_range:
                total = content_range.rsplit("/", 1)[1]
                if total.isdigit():
                    return int(total)
            length = response.headers.get("Content-Length")
            if length and length.isdigit():
                return int(length)
    except Exception:  # noqa: BLE001
        return None
    return None


def _finalize_part(part, dest, sha256=None):
    """Move `part` → `dest`, tolerating Windows antivirus file locks."""
    if sha256 is not None and _file_sha256(part) != sha256:
        raise RuntimeError(
            f"downloaded file failed sha256 (expected {sha256[:12]}…)")
    last_error = None
    for attempt in range(1, 8):
        try:
            os.replace(part, dest)
            return dest
        except OSError as error:  # noqa: BLE001
            last_error = error
            winerr = getattr(error, "winerror", None)
            if winerr == 32 or getattr(error, "errno", None) in (13, 16):
                time.sleep(0.4 * attempt)
                continue
            break
    # Last resort: copy then delete (works when replace is locked).
    import shutil
    try:
        shutil.copyfile(part, dest)
        try:
            os.remove(part)
        except OSError:
            pass
        return dest
    except OSError as error:
        raise RuntimeError(
            f"could not finalize {part} → {dest}: {last_error or error}"
        ) from error


def _download(url, dest, retries=4, sha256=None):
    """Fetches `url` to `dest`, with resume support for multi‑GB archives.

    Partial files are kept as `dest.part` so a dropped link can continue.
    Optional `sha256` rejects corrupt cached files and re-downloads them.
    """
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        if sha256 is None or _file_sha256(dest) == sha256:
            return dest
        print(f"  cached {os.path.basename(dest)} failed checksum; "
              "re-downloading", flush=True)
        try:
            os.remove(dest)
        except OSError:
            pass
    os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
    part = dest + ".part"
    # A prior attempt may have finished the body but failed on os.replace
    # (common on Windows when Defender briefly locks the .part).
    if os.path.exists(part) and os.path.getsize(part) > 0:
        expected = _remote_content_length(url)
        size = os.path.getsize(part)
        if expected and size >= expected:
            if size > expected:
                with open(part, "rb+") as handle:
                    handle.truncate(expected)
            try:
                _finalize_part(part, dest, sha256=sha256)
                print(f"  saved {os.path.basename(dest)} "
                      f"({os.path.getsize(dest) / 1e9:.2f} GB) "
                      f"[finalized existing .part]", flush=True)
                return dest
            except Exception as error:  # noqa: BLE001
                print(f"  could not finalize existing .part: {error}",
                      flush=True)

    print(f"downloading {os.path.basename(dest)} ...", flush=True)
    debug_log("download", "started", url=url, dest=dest)
    system_proxies = urllib.request.getproxies()
    if _proxy_is_localhost(system_proxies):
        print(f"  note: system proxy is {system_proxies} — trying proxy first, "
              f"then direct on SSL/connection failure (GCS may 403 without VPN)",
              flush=True)

    last_error = None
    known_total = None
    for attempt in range(1, retries + 1):
        # Prefer the system proxy (needed for geo-blocked GCS). Only force a
        # direct attempt after a prior failure, and not on every even try — a
        # bare direct GET to tips_data returns HTTP 403 on this machine.
        force_direct = (
            _proxy_is_localhost(system_proxies)
            and attempt >= 3
            and last_error is not None
            and _should_bypass_proxy(last_error)
            and not _is_forbidden(last_error)
        )
        existing = os.path.getsize(part) if os.path.exists(part) else 0
        # HTTP 416: .part is already past EOF (complete or corrupt oversized).
        if existing > 0 and (
                getattr(last_error, "code", None) == 416
                or "416" in _proxy_failure_text(last_error or "")):
            expected = known_total or _remote_content_length(
                url, force_direct=force_direct)
            if expected and existing >= expected:
                if existing > expected:
                    with open(part, "rb+") as handle:
                        handle.truncate(expected)
                _finalize_part(part, dest, sha256=sha256)
                print(f"  saved {os.path.basename(dest)} "
                      f"({os.path.getsize(dest) / 1e9:.2f} GB) "
                      f"[recovered from HTTP 416]", flush=True)
                return dest
            print("  HTTP 416 with incomplete/unknown size — restarting "
                  "download from byte 0", flush=True)
            try:
                os.remove(part)
            except OSError:
                pass
            existing = 0

        headers = {"User-Agent": "tips-clip-ad/1.0 (research; weight fetch)"}
        if existing > 0:
            headers["Range"] = f"bytes={existing}-"
            print(f"  resume attempt {attempt}/{retries} from "
                  f"{existing / 1e9:.2f} GB"
                  f"{' [direct]' if force_direct else ''}", flush=True)
        elif force_direct:
            print(f"  attempt {attempt}/{retries} [direct]", flush=True)
        request = urllib.request.Request(url, headers=headers)
        try:
            with _open_url(request, timeout=180, force_direct=force_direct) as response:
                status = getattr(response, "status", 200)
                # 200 = full body; 206 = resumed range
                if status == 200 and existing > 0:
                    existing = 0  # server ignored Range; rewrite
                total_header = response.headers.get("Content-Length")
                content_range = response.headers.get("Content-Range")
                if content_range and "/" in content_range:
                    total = int(content_range.rsplit("/", 1)[1])
                elif total_header and status == 200:
                    total = int(total_header)
                elif total_header and existing > 0:
                    total = existing + int(total_header)
                else:
                    total = 0
                if total > 0:
                    known_total = total
                done = existing
                last_pct = -1 if total <= 0 else (100 * done // total)
                mode = "wb" if existing == 0 else "ab"
                with open(part, mode) as handle:
                    while True:
                        chunk = response.read(1024 * 1024)
                        if not chunk:
                            break
                        handle.write(chunk)
                        done += len(chunk)
                        if total > 0:
                            pct = 100 * done // total
                            if pct >= last_pct + 5:
                                print(f"  {pct}% of {total / 1e9:.2f} GB",
                                      flush=True)
                                last_pct = pct
                        elif done % (200 * 1024 * 1024) < 1024 * 1024:
                            print(f"  {done / 1e9:.2f} GB downloaded...",
                                  flush=True)
            size = os.path.getsize(part) if os.path.exists(part) else 0
            if size <= 0:
                raise RuntimeError("download produced an empty file")
            if total > 0 and size < total:
                raise RuntimeError(
                    f"incomplete download ({size} < {total} bytes)")
            _finalize_part(part, dest, sha256=sha256)
            print(f"  saved {os.path.basename(dest)} "
                  f"({os.path.getsize(dest) / 1e9:.2f} GB)", flush=True)
            return dest
        except Exception as error:  # noqa: BLE001
            last_error = error
            debug_log("download", "attempt failed", url=url, attempt=attempt,
                      error=str(error), force_direct=force_direct)
            print(f"  attempt {attempt}/{retries} failed: {error}", flush=True)
            # Mid-stream SSL EOF often leaves a truncated .part; keep it for
            # resume unless the server clearly ignored Range last time.
            time.sleep(min(10, 2 * attempt))

    raise RuntimeError(
        f"could not download {url}\n"
        f"  -> last error: {last_error}\n"
        f"  -> If you see WinError 10061 / SSL UNEXPECTED_EOF: start your local "
        f"proxy (v2rayN on 127.0.0.1:10809) — downloads retry direct after proxy "
        f"failures, but GCS tips_data often returns HTTP 403 without the VPN.\n"
        f"  -> Or fetch the file manually into {os.path.dirname(dest)}"
    )


# Checkpoints are fetched lazily by TipsBackbone the first time it is built, so
# a notebook that never touches TIPS never pays for a multi-gigabyte download
# and a network problem surfaces at the backbone, not at import time.
TIPS_VISION_CKPT = TIPS_TEXT_CKPT = TIPS_TOKENIZER = None

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
for _directory in (ARTIFACT_DIR, CHECKPOINT_DIR, TABLE_DIR, COMPLEXITY_DIR):
    os.makedirs(_directory, exist_ok=True)

print(f"torch {torch.__version__} on {DEVICE.upper()}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))
print(f"platform={PLATFORM} | project root {PROJECT_ROOT}")
print(f"weights={WEIGHTS_DIR} | tips_ckpt={TIPS_CKPT_DIR}")
print(f"pkg_resources: {PKG_RESOURCES_STATUS} | TIPS source: "
      f"{'ready' if TIPS_SOURCE_AVAILABLE else 'unavailable'}"
      f" | system_proxy={urllib.request.getproxies() or '{}'}")
if MISSING_DEPENDENCIES:
    print(f"  [warn] still missing: {sorted(MISSING_DEPENDENCIES)} -- the "
          "backbones that need them will be skipped")
print(f"MODELS={MODELS} | TIPS default {TIPS_VERSION}-{TIPS_VARIANT} | "
      f"CLIP {CLIP_BACKBONE} | SigLIP2 {SIGLIP2_MODEL} | "
      f"DINOv2.txt hub entry from model_configs/ | input {INPUT_SIZE}px")

## Cell 3 — Determinism (seed 111)

Seeds Python, NumPy and Torch, and pins cuDNN to deterministic kernels.

The important function here is `derived_seed`. Several corruptions draw random
fields — noise, glass displacement, the direction of a geometric warp. If those
were drawn from the global RNG stream, an image's corruption would depend on how
many images had been processed before it, and a sweep resumed after a Colab
disconnect would silently produce *different data* from an uninterrupted run.
Instead, each corruption's seed is derived by hashing the identifying strings
(seed, dataset, category, file name, corruption, severity), so an image always
receives the same corruption no matter when it is reached.

In [ ]:
# =============================================================================
# CELL 3 -- Determinism
# =============================================================================
import hashlib
import random

import numpy as np
import torch


def seed_everything(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def derived_seed(*parts):
    """A stable seed for a named piece of work, independent of execution order."""
    key = "|".join(str(part) for part in (SEED, *parts)).encode("utf-8")
    return int.from_bytes(hashlib.blake2b(key, digest_size=4).digest(), "big")


def dataloader_kwargs(seed=SEED):
    generator = torch.Generator()
    generator.manual_seed(seed)

    def worker_init_fn(worker_id):
        random.seed(seed + worker_id)
        np.random.seed(seed + worker_id)

    return {"generator": generator, "worker_init_fn": worker_init_fn}


seed_everything(SEED)
print(f"seeded python, numpy and torch with {SEED}; cuDNN set to deterministic")

## Cell 4 — Datasets: acquisition and test-split indices

Resolves `MVTEC_ROOT` / `VISA_ROOT` (project-local unpacks preferred — e.g.
`VisA_20220922/` and `mvtec_anomaly_detection/` next to the notebook), then builds
the **test-split** index for each category.

When VisA is already project-local, `DOWNLOAD_IF_MISSING` defaults to **False** so
a manual MVTec drop is not raced by a Hugging Face download into `DATA_ROOT`
(e.g. `E:\tips_clip_ad_data`). Place the extracted folder at
`<project>/mvtec_anomaly_detection` (same level as VisA) and re-run this cell.
Incomplete folders or a local `.tar.xz` / `.part` are treated as in-progress —
no second download. To force a fetch into the project-local path, set
`DOWNLOAD_IF_MISSING = True` (archive lands beside `MVTEC_ROOT`, not on `E:\`).

Both readers return the same record shape, so every later cell is
dataset-agnostic:

```python
{"image": path, "mask": path or None, "label": 0 normal / 1 anomalous}
```

`mask=None` means a known-normal image, i.e. an all-zero ground-truth mask.

For VisA the official `split_csv/1cls.csv` is read directly rather than
reorganising the directory tree — that is the one-class split VisA is benchmarked
under, and reading it avoids the copy step most reference implementations
require. The cell finishes by verifying every indexed file actually exists, which
catches a truncated download before the sweep starts rather than eight hours in.

Expected totals: MVTec 15 categories / 1725 test images (467 normal, 1258
anomalous); VisA 12 categories / 2162 test images (962 normal, 1200 anomalous).

In [ ]:
# =============================================================================
# CELL 4 -- Dataset acquisition and test-split indices
# =============================================================================
import csv
import shutil
import tarfile

# Official mydrive.ch share links rot frequently (HTTP 404). Keep a short list of
# working mirrors and try them in order. Validated 2026-08-09:
#   MVTec anomalib mydrive + Hugging Face micguida1  -> HTTP 206, xz magic OK
#   VisA Amazon S3                                   -> HTTP 206, tar OK
MVTEC_URLS = (
    # Community Hugging Face mirror of the official .tar.xz (~5.26 GB) — usually
    # the fastest reliable host (mydrive.ch share links often 404 or crawl).
    "https://huggingface.co/datasets/micguida1/mvtech_anomaly_detection/"
    "resolve/main/mvtec_anomaly_detection.tar.xz",
    # Anomalib's current mydrive share (same archive; can be slow).
    "https://www.mydrive.ch/shares/150996/b52ecdcbf521176e9db9c731f2304b27/"
    "download/420938113-1629960298/mvtec_anomaly_detection.tar.xz",
    # Legacy share (often 404; kept last for completeness).
    "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f282"
    "/download/420938113-1629952094/mvtec_anomaly_detection.tar.xz",
)
VISA_URLS = (
    "https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar",
)
MVTEC_URL = MVTEC_URLS[0]
VISA_URL = VISA_URLS[0]
# Provisional default. After resolve_dataset_roots(), if VisA is already
# project-local we flip this to False so a manual MVTec drop next to VisA is
# not raced by a Hugging Face download into DATA_ROOT / E:\...
# Override anytime: DOWNLOAD_IF_MISSING = True  (downloads into MVTEC_ROOT parent)
# or set env TIPS_DOWNLOAD_IF_MISSING=1 / 0 before running this cell.
_env_dl = os.environ.get("TIPS_DOWNLOAD_IF_MISSING", "").strip().lower()
if _env_dl in ("1", "true", "yes", "on"):
    DOWNLOAD_IF_MISSING = True
elif _env_dl in ("0", "false", "no", "off"):
    DOWNLOAD_IF_MISSING = False
else:
    DOWNLOAD_IF_MISSING = True
del _env_dl

MVTEC_CATEGORIES = (
    "bottle", "cable", "capsule", "carpet", "grid", "hazelnut", "leather",
    "metal_nut", "pill", "screw", "tile", "toothbrush", "transistor", "wood",
    "zipper",
)
VISA_CATEGORIES = (
    "candle", "capsules", "cashew", "chewinggum", "fryum", "macaroni1",
    "macaroni2", "pcb1", "pcb2", "pcb3", "pcb4", "pipe_fryum",
)
CATEGORIES = {"mvtec": MVTEC_CATEGORIES, "visa": VISA_CATEGORIES}
DATASET_ROOTS = {"mvtec": MVTEC_ROOT, "visa": VISA_ROOT}

# MVTec category names carry an underscore that reads badly inside a prompt.
PROMPT_CLASS_NAMES = {"metal_nut": "metal nut", "pipe_fryum": "pipe fryum",
                      "pcb1": "printed circuit board", "pcb2": "printed circuit board",
                      "pcb3": "printed circuit board", "pcb4": "printed circuit board",
                      "macaroni1": "macaroni", "macaroni2": "macaroni"}


def prompt_class_name(category):
    return PROMPT_CLASS_NAMES.get(category, category.replace("_", " "))


def _extract(archive, dest):
    print(f"extracting {os.path.basename(archive)} ...", flush=True)
    with tarfile.open(archive) as tar:
        # Python 3.14+ warns / will require an extraction filter; "data" is safe
        # for these public dataset archives.
        try:
            tar.extractall(dest, filter="data")
        except TypeError:
            tar.extractall(dest)


def _download_first(urls, dest):
    """Try each URL until one lands a non-empty file at `dest`."""
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        print(f"using cached {os.path.basename(dest)} "
              f"({os.path.getsize(dest) / 1e9:.2f} GB)", flush=True)
        return dest
    errors = []
    for url in urls:
        print(f"trying {url}", flush=True)
        try:
            return _download(url, dest)
        except Exception as error:  # noqa: BLE001
            errors.append(f"{url}\n    -> {error}")
            print(f"  failed: {error}", flush=True)
            if os.path.exists(dest + ".part"):
                try:
                    os.remove(dest + ".part")
                except OSError:
                    pass
    raise RuntimeError(
        "could not download dataset archive; every mirror failed:\n  - "
        + "\n  - ".join(errors)
        + f"\nPlace the archive manually at {dest} and re-run this cell."
    )


def _looks_like_visa(root):
    return os.path.isfile(os.path.join(root, "split_csv", "1cls.csv"))


def _looks_like_mvtec(root):
    return os.path.isdir(os.path.join(root, "bottle", "test"))


def _dataset_project_bases():
    """Bases where the user commonly unpacks VisA / MVTec (next to the notebook)."""
    _project = PROJECT_ROOT if "PROJECT_ROOT" in globals() else None
    return tuple(
        b for b in (
            _project,
            WORKSPACE_ROOT,
            os.getcwd(),
            os.path.join(WORKSPACE_ROOT, "tips_clip_ad") if WORKSPACE_ROOT else None,
        ) if b
    )


def _path_under_any(path, bases):
    if not path:
        return None
    path_norm = os.path.normcase(os.path.normpath(path))
    for base in bases:
        if not base:
            continue
        base_norm = os.path.normcase(os.path.normpath(base))
        if path_norm == base_norm or path_norm.startswith(base_norm + os.sep):
            return base
    return None


def _mvtec_archive_paths(mvtec_root=None, include_data_root=True):
    """Candidate locations for mvtec_anomaly_detection.tar.xz (.part = in progress)."""
    root = mvtec_root or MVTEC_ROOT
    names = ("mvtec_anomaly_detection.tar.xz",)
    bases = []
    if root:
        bases.append(os.path.dirname(root))
        bases.append(root)
    bases.extend(_dataset_project_bases())
    if include_data_root:
        bases.append(DATA_ROOT)
    out, seen = [], set()
    for base in bases:
        if not base:
            continue
        for name in names:
            path = os.path.join(base, name)
            key = os.path.normcase(os.path.normpath(path))
            if key in seen:
                continue
            seen.add(key)
            out.append(path)
    return out


def _mvtec_manual_in_progress(mvtec_root=None):
    """True when a project-local MVTec drop / extract is underway (do not HF-download).

    Only inspects project-local / MVTEC_ROOT-adjacent paths. An orphaned
    DATA_ROOT/*.part from an older run must not block forever.
    """
    root = mvtec_root or MVTEC_ROOT
    reasons = []
    if root and os.path.isdir(root) and not _looks_like_mvtec(root):
        reasons.append(f"incomplete folder at {root} (no bottle/test yet)")
    for archive in _mvtec_archive_paths(root, include_data_root=False):
        part = archive + ".part"
        if os.path.exists(part):
            size = os.path.getsize(part)
            reasons.append(
                f"partial archive {part} ({size / 1e9:.2f} GB) — download still running")
        elif os.path.isfile(archive) and os.path.getsize(archive) > 0:
            if not (root and _looks_like_mvtec(root)):
                reasons.append(f"archive present at {archive} "
                               f"({os.path.getsize(archive) / 1e9:.2f} GB)")
    return reasons


def resolve_dataset_roots():
    """Point MVTEC_ROOT / VISA_ROOT at whatever local layout the user unpacked.

    VisA discovery order (keep official extract name; do not require rename):
      1. <PROJECT_ROOT>/VisA_20220922, visa_20220922, VisA
      2. <WORKSPACE_ROOT> / cwd variants of the same names
      3. <DATA_ROOT>/VisA_20220922, visa_20220922, VisA
      4. legacy flat extract (<PROJECT_ROOT>/split_csv/1cls.csv) or prior VISA_ROOT

    MVTec discovery order (project-local BEFORE DATA_ROOT / E:\\...):
      1. <PROJECT_ROOT>/mvtec_anomaly_detection, MVTec_AD, mvtec
      2. <WORKSPACE_ROOT> / cwd variants of the same names
      3. When VisA is project-local → intended MVTEC_ROOT alongside VisA
      4. <DATA_ROOT>/... only if VisA is not project-local
    Incomplete project-local folders keep the intended path; fetch_mvtec waits.
    """
    global MVTEC_ROOT, VISA_ROOT, DATASET_ROOTS

    _visa_names = ("VisA_20220922", "visa_20220922", "VisA")
    _mvtec_names = ("mvtec_anomaly_detection", "MVTec_AD", "mvtec")
    _project_bases = _dataset_project_bases()
    visa_candidates = []
    for base in _project_bases:
        if not base:
            continue
        for name in _visa_names:
            visa_candidates.append(os.path.join(base, name))
    for name in _visa_names:
        visa_candidates.append(os.path.join(DATA_ROOT, name))
    for base in _project_bases:
        if base:
            visa_candidates.append(base)  # flat extract (split_csv + candle/...)
    visa_candidates.append(VISA_ROOT)

    seen = set()
    for candidate in visa_candidates:
        if not candidate:
            continue
        key = os.path.normcase(os.path.normpath(candidate))
        if key in seen:
            continue
        seen.add(key)
        if _looks_like_visa(candidate):
            if os.path.normpath(candidate) != os.path.normpath(VISA_ROOT):
                print(f"VisA detected at {candidate} (was looking in {VISA_ROOT})",
                      flush=True)
            VISA_ROOT = candidate
            break

    mvtec_candidates = []
    for base in _project_bases:
        if not base:
            continue
        for name in _mvtec_names:
            mvtec_candidates.append(os.path.join(base, name))

    seen = set()
    _mvtec_resolved = False
    for candidate in mvtec_candidates:
        if not candidate:
            continue
        key = os.path.normcase(os.path.normpath(candidate))
        if key in seen:
            continue
        seen.add(key)
        if _looks_like_mvtec(candidate):
            if os.path.normpath(candidate) != os.path.normpath(MVTEC_ROOT):
                print(f"MVTec detected at {candidate} (was looking in {MVTEC_ROOT})",
                      flush=True)
            MVTEC_ROOT = candidate
            _mvtec_resolved = True
            break

    # Is VisA under a project base? If so, keep MVTec intended alongside it and
    # do not silently switch to DATA_ROOT / E:\ while a local download lands.
    _visa_project_base = None
    visa_norm = os.path.normcase(os.path.normpath(VISA_ROOT))
    for base in _project_bases:
        if not base:
            continue
        base_norm = os.path.normcase(os.path.normpath(base))
        if visa_norm == base_norm or visa_norm.startswith(base_norm + os.sep):
            _visa_project_base = base
            break

    if not _mvtec_resolved and _visa_project_base is None:
        for name in _mvtec_names:
            candidate = os.path.join(DATA_ROOT, name)
            key = os.path.normcase(os.path.normpath(candidate))
            if key in seen:
                continue
            seen.add(key)
            if _looks_like_mvtec(candidate):
                if os.path.normpath(candidate) != os.path.normpath(MVTEC_ROOT):
                    print(f"MVTec detected at {candidate} "
                          f"(was looking in {MVTEC_ROOT})", flush=True)
                MVTEC_ROOT = candidate
                _mvtec_resolved = True
                break
        if not _mvtec_resolved and MVTEC_ROOT and _looks_like_mvtec(MVTEC_ROOT):
            _mvtec_resolved = True

    if not _mvtec_resolved:
        # Prefer an existing (possibly incomplete) project-local folder.
        for base in _project_bases:
            if not base:
                continue
            for name in _mvtec_names:
                candidate = os.path.join(base, name)
                if os.path.isdir(candidate):
                    if os.path.normpath(candidate) != os.path.normpath(MVTEC_ROOT):
                        print(f"MVTec folder (possibly incomplete) at {candidate}",
                              flush=True)
                    MVTEC_ROOT = candidate
                    _mvtec_resolved = True
                    break
            if _mvtec_resolved:
                break

    if not _mvtec_resolved and _visa_project_base is not None:
        intended = os.path.join(_visa_project_base, "mvtec_anomaly_detection")
        if os.path.normpath(intended) != os.path.normpath(MVTEC_ROOT):
            print(f"MVTec intended at {intended} (alongside VisA; was {MVTEC_ROOT})",
                  flush=True)
        MVTEC_ROOT = intended
        _mvtec_resolved = True

    if not _mvtec_resolved:
        # Last resort: keep prior MVTEC_ROOT, or DATA_ROOT default.
        if not MVTEC_ROOT:
            MVTEC_ROOT = os.path.join(DATA_ROOT, "mvtec_anomaly_detection")

    DATASET_ROOTS = {"mvtec": MVTEC_ROOT, "visa": VISA_ROOT}


def _extract_mvtec_archive(archive, dest):
    os.makedirs(dest, exist_ok=True)
    _extract(archive, dest)
    if not _looks_like_mvtec(dest):
        nested = os.path.join(dest, "mvtec_anomaly_detection")
        if _looks_like_mvtec(nested):
            for name in os.listdir(nested):
                src = os.path.join(nested, name)
                dst = os.path.join(dest, name)
                if not os.path.exists(dst):
                    os.rename(src, dst)
    marker = os.path.join(dest, "bottle", "test")
    if not _looks_like_mvtec(dest):
        raise FileNotFoundError(
            f"MVTec extract finished but {marker} is missing under {dest}")


def fetch_mvtec():
    """Locate / wait for / optionally download MVTec into MVTEC_ROOT.

    Never starts a competing Hugging Face download while a project-local folder
    or archive (.tar.xz / .part) indicates a manual transfer is in progress.
    Auto-download (when allowed) writes the archive next to MVTEC_ROOT — not
    under DATA_ROOT / E:\\ — and extracts into MVTEC_ROOT.
    """
    resolve_dataset_roots()
    if _looks_like_mvtec(MVTEC_ROOT):
        print(f"MVTec already present under {MVTEC_ROOT}", flush=True)
        return

    marker = os.path.join(MVTEC_ROOT, "bottle", "test")
    visa_project = _path_under_any(VISA_ROOT, _dataset_project_bases())
    mvtec_project = _path_under_any(MVTEC_ROOT, _dataset_project_bases())

    # Prefer extracting a finished local archive over any network download.
    for archive in _mvtec_archive_paths(MVTEC_ROOT, include_data_root=True):
        part = archive + ".part"
        if os.path.exists(part):
            continue
        if os.path.isfile(archive) and os.path.getsize(archive) > 1_000_000:
            print(f"extracting local MVTec archive {archive} -> {MVTEC_ROOT}",
                  flush=True)
            _extract_mvtec_archive(archive, MVTEC_ROOT)
            print(f"MVTec ready under {MVTEC_ROOT}", flush=True)
            return

    in_progress = _mvtec_manual_in_progress(MVTEC_ROOT)
    if in_progress:
        detail = "; ".join(in_progress)
        print(f"[wait] MVTec manual download/extract in progress — "
              f"NOT starting Hugging Face download.\n"
              f"  intended MVTEC_ROOT={MVTEC_ROOT}\n"
              f"  signals: {detail}\n"
              f"  Re-run this cell when {marker} exists.",
              flush=True)
        raise FileNotFoundError(
            f"waiting for manual MVTec at {MVTEC_ROOT} ({detail})")

    if os.path.isdir(MVTEC_ROOT) and not _looks_like_mvtec(MVTEC_ROOT):
        print(f"[wait] MVTec folder exists but is incomplete at {MVTEC_ROOT} "
              f"(expected {marker}). NOT downloading.", flush=True)
        raise FileNotFoundError(
            f"MVTec folder exists at {MVTEC_ROOT} but is incomplete "
            f"(expected {marker}). Wait for the download/extract to finish.")

    # VisA already sits next to the notebook → expect MVTec there too; do not
    # race a manual browser download with an automatic HF pull into E:\.
    if visa_project is not None and mvtec_project is not None and not DOWNLOAD_IF_MISSING:
        print(f"[wait] VisA is project-local; place MVTec at\n"
              f"  {MVTEC_ROOT}\n"
              f"  (folder name mvtec_anomaly_detection next to VisA). "
              f"NOT auto-downloading.\n"
              f"  When finished, re-run this cell. "
              f"To force a fetch into that path, set "
              f"DOWNLOAD_IF_MISSING=True and re-run.",
              flush=True)
        raise FileNotFoundError(
            f"MVTec not found under {MVTEC_ROOT}; waiting for manual placement "
            f"alongside VisA")

    if visa_project is not None and mvtec_project is not None and DOWNLOAD_IF_MISSING:
        print(f"DOWNLOAD_IF_MISSING=True — fetching MVTec into project-local "
              f"{MVTEC_ROOT}", flush=True)

    if not DOWNLOAD_IF_MISSING:
        raise FileNotFoundError(f"MVTec not found under {MVTEC_ROOT}")

    # Archive lands beside MVTEC_ROOT (project path), never forced onto E:\DATA_ROOT.
    archive_dir = os.path.dirname(MVTEC_ROOT) or DATA_ROOT
    os.makedirs(archive_dir, exist_ok=True)
    archive = os.path.join(archive_dir, "mvtec_anomaly_detection.tar.xz")
    print(f"MVTec archive target: {archive}\n"
          f"MVTec extract target: {MVTEC_ROOT}", flush=True)
    _download_first(MVTEC_URLS, archive)
    _extract_mvtec_archive(archive, MVTEC_ROOT)
    print(f"MVTec ready under {MVTEC_ROOT}", flush=True)


def fetch_visa():
    global VISA_ROOT
    # Prefer an already-valid tree (e.g. PROJECT_ROOT/VisA_20220922) — do not
    # rename that folder to VisA, and do not re-download when present.
    resolve_dataset_roots()
    if _looks_like_visa(VISA_ROOT):
        DATASET_ROOTS["visa"] = VISA_ROOT
        print(f"VisA already present under {VISA_ROOT}", flush=True)
        return
    if not DOWNLOAD_IF_MISSING:
        raise FileNotFoundError(f"VisA not found under {VISA_ROOT}")
    archive = os.path.join(DATA_ROOT, "VisA_20220922.tar")
    _download_first(VISA_URLS, archive)
    _extract(archive, DATA_ROOT)
    # Keep the official extract directory name; point VISA_ROOT at it.
    unpacked = os.path.join(DATA_ROOT, "VisA_20220922")
    if _looks_like_visa(unpacked):
        VISA_ROOT = unpacked
    elif _looks_like_visa(os.path.join(DATA_ROOT, "VisA")):
        VISA_ROOT = os.path.join(DATA_ROOT, "VisA")
    DATASET_ROOTS["visa"] = VISA_ROOT
    if not _looks_like_visa(VISA_ROOT):
        raise FileNotFoundError(
            f"VisA extract finished but split_csv/1cls.csv missing under {VISA_ROOT}")


def index_mvtec_test(category):
    root = os.path.join(MVTEC_ROOT, category)
    records = []
    for defect in sorted(os.listdir(os.path.join(root, "test"))):
        image_dir = os.path.join(root, "test", defect)
        if not os.path.isdir(image_dir):
            continue
        for name in sorted(os.listdir(image_dir)):
            if not name.lower().endswith(".png"):
                continue
            stem = os.path.splitext(name)[0]
            records.append({
                "image": os.path.join(image_dir, name),
                "mask": None if defect == "good" else os.path.join(
                    root, "ground_truth", defect, f"{stem}_mask.png"),
                "label": int(defect != "good"),
            })
    return records


def index_visa_test(category):
    """Reads VisA's official one-class split; no directory reorganisation needed."""
    records = []
    with open(os.path.join(VISA_ROOT, "split_csv", "1cls.csv"),
              newline="", encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            if row["object"] != category or row["split"] != "test":
                continue
            is_anomalous = row["label"].strip().lower() != "normal"
            mask = row["mask"].strip()
            records.append({
                "image": os.path.join(VISA_ROOT, row["image"].strip()),
                "mask": os.path.join(VISA_ROOT, mask) if (is_anomalous and mask) else None,
                "label": int(is_anomalous),
            })
    return sorted(records, key=lambda record: record["image"])


INDEXERS = {"mvtec": index_mvtec_test, "visa": index_visa_test}
_INDEX_CACHE = {}


def index_test_split(dataset, category):
    if (dataset, category) in _INDEX_CACHE:
        return _INDEX_CACHE[(dataset, category)]
    records = INDEXERS[dataset](category)
    if not records:
        raise RuntimeError(f"no test images found for {dataset}/{category}")
    missing = [r for r in records
               if not os.path.isfile(r["image"])
               or (r["mask"] and not os.path.isfile(r["mask"]))]
    if missing:
        raise FileNotFoundError(f"{len(missing)} missing file(s) for "
                                f"{dataset}/{category}, first: {missing[0]}")
    _INDEX_CACHE[(dataset, category)] = records
    return records


def relative_paths(dataset, category):
    root = DATASET_ROOTS[dataset]
    return [os.path.relpath(r["image"], root)
            for r in index_test_split(dataset, category)]


# Acquire each dataset independently so one dead mirror cannot block the other.
_DATA_READY = True
_DATA_ERRORS = {}
os.makedirs(DATA_ROOT, exist_ok=True)

# Honour a VisA / MVTec tree the user already unpacked (including flat into the repo).
resolve_dataset_roots()

# When VisA is already project-local, default to NOT auto-fetching MVTec (manual
# drop expected at PROJECT_ROOT/mvtec_anomaly_detection). Env TIPS_DOWNLOAD_IF_MISSING
# wins if set; otherwise leave an explicit True only for fresh Colab/Kaggle setups.
_visa_is_project_local = _path_under_any(VISA_ROOT, _dataset_project_bases()) is not None
_env_dl = os.environ.get("TIPS_DOWNLOAD_IF_MISSING", "").strip().lower()
if _env_dl not in ("1", "true", "yes", "on", "0", "false", "no", "off"):
    if _visa_is_project_local and _looks_like_visa(VISA_ROOT):
        DOWNLOAD_IF_MISSING = False
del _env_dl

print(f"DATA_ROOT={DATA_ROOT}  DOWNLOAD_IF_MISSING={DOWNLOAD_IF_MISSING}", flush=True)
print(f"MVTEC_ROOT={MVTEC_ROOT}", flush=True)
print(f"VISA_ROOT ={VISA_ROOT}", flush=True)
try:
    _free_gb = shutil.disk_usage(DATA_ROOT).free / (1024 ** 3)
    print(f"free disk at DATA_ROOT: {_free_gb:.1f} GB "
          f"(MVTec archive ~5.3 GB + VisA ~1.9 GB; extract needs more)", flush=True)
    if _free_gb < 20:
        print("[warn] low disk space — set DATA_ROOT to a larger drive, e.g.\n"
              "  DATA_ROOT = r'E:\\\\tips_clip_ad_data'", flush=True)
except Exception:  # noqa: BLE001
    pass
if _visa_is_project_local and not DOWNLOAD_IF_MISSING and not _looks_like_mvtec(MVTEC_ROOT):
    print("[note] VisA is project-local and DOWNLOAD_IF_MISSING=False — "
          "cell will wait for manual MVTec at MVTEC_ROOT (no Hugging Face download).",
          flush=True)
# Orphaned competing download from older cell versions (safe to delete).
_legacy_part = os.path.join(DATA_ROOT, "mvtec_anomaly_detection.tar.xz.part")
if os.path.exists(_legacy_part) and _path_under_any(MVTEC_ROOT, _dataset_project_bases()):
    print(f"[note] leftover partial download at {_legacy_part} "
          f"({os.path.getsize(_legacy_part) / 1e9:.2f} GB) is NOT used; "
          f"MVTec target is {MVTEC_ROOT}. You can delete that .part file.",
          flush=True)

if "mvtec" in DATASETS:
    try:
        fetch_mvtec()
    except Exception as _data_error:  # noqa: BLE001
        _DATA_ERRORS["mvtec"] = str(_data_error)
        print(f"[warn] MVTec acquisition failed: {_data_error}", flush=True)

if "visa" in DATASETS:
    try:
        fetch_visa()
    except Exception as _data_error:  # noqa: BLE001
        _DATA_ERRORS["visa"] = str(_data_error)
        print(f"[warn] VisA acquisition failed: {_data_error}", flush=True)

# Re-resolve after fetch/rename so DATASET_ROOTS stay accurate.
resolve_dataset_roots()

_present = {
    "mvtec": _looks_like_mvtec(MVTEC_ROOT),
    "visa": _looks_like_visa(VISA_ROOT),
}
_DATA_READY = all(_present[name] for name in DATASETS if name in _present)
if _DATA_ERRORS or not _DATA_READY:
    print("[warn] dataset acquisition incomplete:\n"
          f"  MVTec ({'OK' if _present['mvtec'] else 'MISSING'}): {MVTEC_ROOT}\n"
          f"  VisA  ({'OK' if _present['visa'] else 'MISSING'}): {VISA_ROOT}\n"
          "  Smoke/sweep need both for the full pptx cross-dataset protocol.\n"
          "  VisA-only is enough to index VisA categories after this cell.",
          flush=True)

# Index whatever is present so VisA results can proceed even if MVTec is still missing.
for _dataset in DATASETS:
    if not _present.get(_dataset):
        continue
    _normal = _anomalous = 0
    _ok = True
    for _category in CATEGORIES[_dataset]:
        try:
            _records = index_test_split(_dataset, _category)
        except Exception as _index_error:  # noqa: BLE001
            print(f"[warn] could not index {_dataset}/{_category}: {_index_error}")
            _present[_dataset] = False
            _DATA_READY = False
            _ok = False
            break
        _anomalous += sum(r["label"] for r in _records)
        _normal += sum(1 - r["label"] for r in _records)
    if _ok:
        print(f"{_dataset}: {len(CATEGORIES[_dataset])} categories, "
              f"{_normal + _anomalous} test images "
              f"({_normal} normal / {_anomalous} anomalous)  "
              f"root={DATASET_ROOTS[_dataset]}")

# Allow smoke when the first eval dataset in PROTOCOL is present (usually VisA).
_SMOKE_DATASET = PROTOCOL[0][1]
_DATA_READY_FOR_SMOKE = bool(_present.get(_SMOKE_DATASET, False))
if _DATA_READY_FOR_SMOKE and not _DATA_READY:
    print(f"[note] full protocol not ready, but {_SMOKE_DATASET} is — "
          "smoke can run on that dataset.", flush=True)

## Cell 5 — The eleven corruptions, applied deterministically

Implements the slide 18/19 selection at severities 1–3, giving
`3 × 11 + 1 clean = 34` evaluation settings per category.

Three decisions worth stating explicitly, because each one would otherwise
quietly bias the comparison:

**1. Corrupted once, shown to both backbones.** Corruption happens at a single
canonical resolution (`INPUT_SIZE`, 518) and the resulting pixels go to both
encoders. Corrupting per backbone would give TIPS and CLIP different blur radii
and different post-resampling noise, and any difference in the results would no
longer be attributable to the encoders.

**2. Geometric warps also transform the mask.** Rotation, magnification and shift
move the defect. The ground-truth mask is warped by the *identical* affine
transform — bilinear for the image, nearest for the mask — so the two stay in
register. Skipping this is the single most damaging mistake available here: pixel
AUROC would collapse for reasons that have nothing to do with the backbone.
Out-of-frame regions are filled by edge replication; zero-filling would paint a
large black border that itself looks anomalous, and reflection would duplicate the
defect into unlabelled locations.

**3. Geometric severities are ours, and labelled as such.** Hendrycks &
Dietterich calibrate severities for the noise, blur and photometric corruptions
(ImageNet-C), but the geometric perturbations come from ImageNet-P, which defines
perturbation *sequences* rather than graded severities. The magnitudes in
`GEOMETRIC_MAGNITUDES` — rotation ±5/10/15°, magnification 1.05/1.10/1.15×, shift
2/4/6% of the side — are therefore a choice of ours, documented in the config so
a reader knows which numbers are standard and which are not. Direction and sign
are drawn per image from the derived seed, so the perturbation is unbiased yet
still exactly reproducible.

One caveat to record: ImageNet-C severities were calibrated at 224×224, and here
they are applied at 518×518, so the blurs are narrower relative to the image than
in the original benchmark. This is applied identically to both backbones, so the
comparison is unaffected, but absolute degradations are not directly comparable
to ImageNet-C literature.

In [ ]:
# =============================================================================
# CELL 5 -- Deterministic corruptions (slide 18/19 selection)
# =============================================================================
import math

import torch.nn.functional as F

# setuptools>=81 removed pkg_resources; the setup cell installs a shim first.
try:
    from imagecorruptions import corrupt as _imagenet_c_corrupt
    from imagecorruptions import get_corruption_names
except Exception as _corrupt_import_error:  # noqa: BLE001
    raise ImportError(
        "imagecorruptions failed to import. Re-run the setup cell "
        f"(pkg_resources shim). Underlying error: {_corrupt_import_error}"
    ) from _corrupt_import_error

CORRUPT_RES = INPUT_SIZE
GEOMETRIC = CORRUPTION_GROUPS.get("geometric", ())
SELECTED_CORRUPTIONS = tuple(name for group in CORRUPTION_GROUPS.values()
                             for name in group)
CORRUPTION_GROUP_OF = {name: group
                       for group, names in CORRUPTION_GROUPS.items()
                       for name in names}
CLEAN = ("clean", 0)

_available = set(get_corruption_names("all"))
_unknown = [name for name in SELECTED_CORRUPTIONS
            if name not in _available and name not in GEOMETRIC]
if _unknown:
    raise ValueError(f"unknown corruption(s) {_unknown}; imagecorruptions offers "
                     f"{sorted(_available)}")


def corruption_grid():
    grid = [CLEAN] if INCLUDE_CLEAN else []
    return grid + [(name, severity)
                   for name in SELECTED_CORRUPTIONS
                   for severity in SEVERITIES]


def _affine_matrix(corruption, severity, rng):
    """The 2x3 matrix mapping output normalised coordinates to input ones."""
    magnitude = GEOMETRIC_MAGNITUDES[corruption][severity - 1]
    if corruption == "rotation":
        angle = math.radians(magnitude) * rng.choice([-1.0, 1.0])
        cos, sin = math.cos(angle), math.sin(angle)
        return [[cos, -sin, 0.0], [sin, cos, 0.0]]
    if corruption == "zoom_scale":
        inverse = 1.0 / magnitude
        return [[inverse, 0.0, 0.0], [0.0, inverse, 0.0]]
    if corruption == "shift":
        # Normalised coordinates span [-1, 1], so a shift of `magnitude` side
        # lengths is 2 * magnitude in grid units.
        dx, dy = (2.0 * magnitude * rng.choice([-1.0, 1.0]) for _ in range(2))
        return [[1.0, 0.0, dx], [0.0, 1.0, dy]]
    raise ValueError(f"{corruption} is not a geometric perturbation")


def _warp(image_uint8, mask_uint8, corruption, severity, seed):
    """Warps image and mask by one shared affine transform, replicating edges."""
    rng = np.random.default_rng(seed)
    theta = torch.tensor([_affine_matrix(corruption, severity, rng)],
                         dtype=torch.float32)
    # PIL hands over a read-only buffer, which torch refuses to wrap.
    image = torch.from_numpy(np.array(image_uint8, copy=True))
    image = image.permute(2, 0, 1)[None].float()
    grid = F.affine_grid(theta, list(image.shape), align_corners=False)
    warped = F.grid_sample(image, grid, mode="bilinear", padding_mode="border",
                           align_corners=False)
    warped_image = (warped[0].permute(1, 2, 0)
                    .round().clamp(0, 255).to(torch.uint8).numpy())

    if mask_uint8 is None or not mask_uint8.any():
        return warped_image, mask_uint8
    mask = torch.from_numpy(np.array(mask_uint8, copy=True))[None, None].float()
    warped_mask = F.grid_sample(mask, grid, mode="nearest",
                                padding_mode="zeros", align_corners=False)
    return warped_image, warped_mask[0, 0].to(torch.uint8).numpy()


def apply_corruption(image_uint8, mask_uint8, corruption, severity, cache_key):
    """Corrupts an HxWx3 uint8 image, and its mask when the warp moves the defect.

    Reproducible by construction: the RNG is seeded from `cache_key` rather than
    drawn from a global stream, so an image gets the same corruption wherever in
    the sweep it is reached and a resumed run matches an uninterrupted one.
    """
    if corruption == "clean" or severity == 0:
        return image_uint8, mask_uint8

    seed = derived_seed(cache_key, corruption, severity)
    if corruption in GEOMETRIC:
        return _warp(image_uint8, mask_uint8, corruption, severity, seed)

    # A mask is meaningless for a photometric or noise corruption, which move no
    # pixels, so only the image changes.
    state = np.random.get_state()
    try:
        np.random.seed(seed)
        corrupted = _imagenet_c_corrupt(image_uint8, corruption_name=corruption,
                                        severity=severity)
    finally:
        np.random.set_state(state)
    return corrupted, mask_uint8


print(f"{len(SELECTED_CORRUPTIONS)} corruptions in {len(CORRUPTION_GROUPS)} groups "
      f"x {len(SEVERITIES)} severities"
      f"{' + clean' if INCLUDE_CLEAN else ''} = {len(corruption_grid())} settings "
      f"per category, applied at {CORRUPT_RES}px")
for _group, _names in CORRUPTION_GROUPS.items():
    print(f"  {_group:<12} {', '.join(_names)}")

## Cell 6 — The dataset object

Yields one category's test split, optionally corrupted. Images come out as
**uint8 at 518×518**, deliberately un-normalised: a single decoded and corrupted
image can then be normalised differently for each backbone (TIPS expects plain
`[0,1]` RGB, CLIP expects its own channel statistics) without being corrupted
twice.

Masks are loaded at full corruption resolution, warped alongside the image when
the corruption is geometric, and only then reduced to `MAP_RES`. Reducing first
would rotate a 64×64 mask, compounding two lots of interpolation error.

`_downsample_mask` has one guard worth pointing out. Area-downsampling a mask can
erase a defect that is only a few pixels wide, turning a positive into a negative
and quietly deflating recall. When that happens the mask falls back to
max-pooling so the defect survives, at the cost of being a pixel or two more
generous.

In [ ]:
# =============================================================================
# CELL 6 -- Test-split dataset with optional corruption
# =============================================================================
from PIL import Image
from torch.utils.data import ConcatDataset, DataLoader, Dataset

Image.MAX_IMAGE_PIXELS = None


def load_mask_full(path, size):
    """Binary mask at `size`, before any downsampling to the metric resolution."""
    if path is None:
        return np.zeros((size, size), dtype=np.uint8)
    mask = Image.open(path).convert("L").resize((size, size), Image.NEAREST)
    return (np.asarray(mask) > 0).astype(np.uint8)


def downsample_mask(mask_uint8, size):
    raw = torch.from_numpy(mask_uint8.astype(np.float32))[None, None]
    reduced = F.interpolate(raw, size=(size, size), mode="area") > 0.5
    if not reduced.any() and raw.any():
        # Preserve defects too small to survive area-downsampling.
        reduced = F.adaptive_max_pool2d(raw, size) > 0
    return reduced[0, 0].to(torch.uint8)


class AnomalyTestSet(Dataset):
    def __init__(self, dataset, category, corruption="clean", severity=0,
                 limit=None):
        self.dataset = dataset
        self.category = category
        self.corruption = corruption
        self.severity = severity
        self.records = index_test_split(dataset, category)
        if limit is not None:
            self.records = self.records[:limit]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        image = Image.open(record["image"]).convert("RGB")
        image = image.resize((CORRUPT_RES, CORRUPT_RES), Image.BICUBIC)
        array = np.asarray(image, dtype=np.uint8)
        mask = load_mask_full(record["mask"], CORRUPT_RES)

        array, mask = apply_corruption(
            array, mask, self.corruption, self.severity,
            cache_key=f"{self.dataset}/{self.category}/"
                      f"{os.path.basename(record['image'])}")

        # PIL exposes its buffer read-only and the clean path returns it
        # unchanged, so copy before handing it to torch.
        array = np.array(array, dtype=np.uint8, copy=True, order="C")
        return {
            "image": torch.from_numpy(array).permute(2, 0, 1),   # uint8 [3,H,W]
            "mask": downsample_mask(mask, MAP_RES),              # uint8 [M,M]
            "label": record["label"],
            "index": index,
        }


def make_loader(dataset, category, corruption="clean", severity=0, limit=None,
                batch_size=None, shuffle=False, num_workers=None):
    data = AnomalyTestSet(dataset, category, corruption, severity, limit)
    return DataLoader(
        data,
        batch_size=batch_size or BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS if num_workers is None else num_workers,
        pin_memory=(DEVICE == "cuda"),
        drop_last=False,
        **dataloader_kwargs(),
    )

## Cell 7 — TIPS backbone

Both backbones expose one interface, so the scoring, prompt and training code is
written exactly once:

| Method | Returns |
| --- | --- |
| `preprocess(uint8 batch)` | normalised float batch at 518px |
| `encode(x)` | `{"object": [B,D], "spatial": [B,D], "dense": {layer: [B,h,w,D]}}` |
| `init_prompt(...)` | initial context embeddings + fixed suffix/padding |
| `encode_text(ctx, aux)` | `[K,D]` text embeddings, differentiable in `ctx` |
| `encode_fixed_text(texts)` | `[K,D]` embeddings for handcrafted prompts |

TIPS specifics:

- **Two global tokens.** TIPS emits an object-centric CLS token and a spatial one.
  DeepMind's own inference script uses the first for text similarity, while
  Tipsomaly's Table 2(c) finds the *spatial* token more discriminative for
  anomaly scoring (MVTec image AUROC 95.3 vs. 93.5). Both are returned and
  `GLOBAL_TOKEN` selects between them.
- **The values trick.** For the deepest selected block, the value projections of
  the attention layer are used as patch descriptors instead of the attention
  output. Patch–text alignment is markedly better that way; this is exactly what
  TIPS's official zero-shot segmentation notebook does.
- **Instantiated at the checkpoint's own positional-embedding grid** (32×32 for
  the HR checkpoints) so the state dict loads strictly, then fed 518px inputs,
  which TIPS interpolates internally.
- **Prompt injection.** TIPS pools its text tower over every non-padding
  position, so all `N_CTX` positions are force-marked valid; otherwise trailing
  context vectors would be masked out and never receive gradient.
- **Temperature.** The logit scale comes from the checkpoint's own contrastive
  temperature (≈0.0042 for TIPS, vs. 0.01 for CLIP) rather than a shared
  constant, since the softmax probability is what gets thresholded.

In [ ]:
# =============================================================================
# CELL 7 -- TIPS backbone
# =============================================================================
import sentencepiece as spm

TIPS_VOCAB_SIZE = 32000


def resolve_dense_layers(model_name, depth):
    """Turns depth fractions into 1-based block indices for a tower of `depth`."""
    fractions = (SHARED_DENSE_LAYERS if SHARED_DENSE_LAYERS is not None
                 else DENSE_LAYER_FRACTIONS[model_name])
    return tuple(sorted({min(depth, max(1, int(round(fraction * depth))))
                         for fraction in fractions}))


def _tips_value_block(block, x):
    """One DINOv2-style block with attention replaced by its value path."""
    normed = block.norm1(x)
    batch, length, channels = normed.shape
    heads = block.attn.num_heads
    qkv = (block.attn.qkv(normed)
           .reshape(batch, length, 3, heads, channels // heads)
           .permute(2, 0, 3, 1, 4))
    values = qkv[2].transpose(1, 2).reshape(batch, length, channels)
    x = x + block.ls1(block.attn.proj(values))
    return x + block.ls2(block.mlp(block.norm2(x)))


class TipsBackbone:
    has_two_global_tokens = True

    def __init__(self, device=None, version=None, variant=None, name=None):
        from tips.pytorch import image_encoder, text_encoder

        self.version = version or TIPS_VERSION
        self.variant = variant or TIPS_VARIANT
        self.name = name or ("tips_v2" if self.version == "v2" else "tips")
        self.device = device or DEVICE
        self.image_size = INPUT_SIZE
        self.patch_size = 14

        vision_url, text_url = tips_checkpoint_urls(self.version, self.variant)
        vision_ckpt = _download(
            vision_url, os.path.join(TIPS_CKPT_DIR, os.path.basename(vision_url)))
        text_ckpt = _download(
            text_url, os.path.join(TIPS_CKPT_DIR, os.path.basename(text_url)))
        tokenizer_path = _download(
            TOKENIZER_URL, os.path.join(TIPS_CKPT_DIR, "tokenizer.model"))

        vision_weights = {key: torch.from_numpy(value) for key, value in
                          dict(np.load(vision_ckpt, allow_pickle=False)).items()}
        native_grid = int(round((vision_weights["pos_embed"].shape[1] - 1) ** 0.5))
        builder = {
            "S": image_encoder.vit_small, "B": image_encoder.vit_base,
            "L": image_encoder.vit_large, "So400m": image_encoder.vit_so400m,
            "g": image_encoder.vit_giant2,
        }[self.variant]
        self.image_model = builder(
            img_size=native_grid * self.patch_size,
            patch_size=self.patch_size,
            ffn_layer="swiglu" if self.variant == "g" else "mlp",
            block_chunks=0,
            init_values=1.0,
            interpolate_antialias=True,
            interpolate_offset=0.0,
        )
        self.image_model.load_state_dict(vision_weights)

        text_weights = {key: torch.from_numpy(value) for key, value in
                        dict(np.load(text_ckpt, allow_pickle=False)).items()}
        temperature = text_weights.pop("temperature_contrastive", None)
        temperature = text_weights.pop("temperature", temperature)
        self.temperature = float(temperature) if temperature is not None else 0.0042
        self.text_model = text_encoder.TextEncoder(
            TIPS_TEXT_CONFIG[self.variant], vocab_size=TIPS_VOCAB_SIZE)
        self.text_model.load_state_dict(text_weights)

        for model in (self.image_model, self.text_model):
            model.eval().to(self.device)
            for parameter in model.parameters():
                parameter.requires_grad_(False)

        self.embed_dim = TIPS_TEXT_CONFIG[self.variant]["hidden_size"]
        self.depth = len(self.image_model.blocks)
        # Layer fractions are keyed by the logical family ("tips"), shared by v1/v2.
        layer_key = "tips" if self.name.startswith("tips") else self.name
        self.layers = resolve_dense_layers(layer_key, self.depth)
        self.num_params = sum(p.numel() for p in self.image_model.parameters()) + \
            sum(p.numel() for p in self.text_model.parameters())
        self.tokenizer = spm.SentencePieceProcessor(model_file=tokenizer_path)

    # --- vision --------------------------------------------------------------
    def preprocess(self, images_uint8):
        """TIPS is trained on plain [0,1] RGB: mean 0, std 1, no channel shift."""
        x = images_uint8.to(self.device, non_blocking=True).float().div_(255.0)
        if x.shape[-1] != self.image_size:
            x = F.interpolate(x, size=(self.image_size, self.image_size),
                              mode="bicubic", align_corners=False, antialias=True)
        return x.clamp_(0.0, 1.0)

    def encode(self, x):
        model = self.image_model
        skip = 1 + model.num_register_tokens
        grid = x.shape[-1] // self.patch_size
        value_layer = max(self.layers)

        hidden = model.prepare_tokens_with_masks(x)
        pre_value, dense = None, {}
        for depth, block in enumerate(model.blocks, start=1):
            if depth == value_layer:
                pre_value = hidden
            hidden = block(hidden)
            if depth in self.layers:
                dense[depth] = model.norm(hidden)[:, skip:]

        normed = model.norm(hidden)
        if USE_VALUE_ATTENTION and pre_value is not None:
            dense[value_layer] = model.norm(
                _tips_value_block(model.blocks[value_layer - 1], pre_value))[:, skip:]

        return {
            "object": normed[:, 0],                 # contrastive CLS token
            "spatial": normed[:, 1],                 # register/spatial token
            "dense": {layer: tokens.reshape(tokens.shape[0], grid, grid, -1)
                      for layer, tokens in dense.items()},
        }

    # --- text ----------------------------------------------------------------
    def _tokenize(self, texts, min_length=0):
        ids = [self.tokenizer.encode(text.lower()) for text in texts]
        length = max(min_length, max(len(sequence) for sequence in ids))
        padded = torch.zeros(len(ids), length, dtype=torch.long)
        paddings = torch.ones(len(ids), length, dtype=torch.float32)
        for row, sequence in enumerate(ids):
            padded[row, :len(sequence)] = torch.tensor(sequence)
            paddings[row, :len(sequence)] = 0.0
        return padded.to(self.device), paddings.to(self.device)

    def init_prompt(self, suffixes):
        """Learnable context followed by a fixed suffix, e.g. [V1..V8] "damaged object".

        The context is initialised from the suffix's own token embeddings so
        optimisation starts inside the text manifold rather than at random noise.
        """
        ids, paddings = self._tokenize(suffixes, min_length=N_CTX + 1)
        with torch.no_grad():
            embeds = self.text_model.token_embedding(ids)
        seed_source = embeds[:, :1].repeat(1, N_CTX, 1)
        ctx = seed_source + 0.02 * torch.randn_like(seed_source)
        # Every context position must count as real, or the mean-pooled text
        # tower would ignore the tail of the context and it would never train.
        paddings = torch.cat([torch.zeros(len(suffixes), N_CTX, device=self.device),
                              paddings], dim=1)
        return ctx, {"paddings": paddings, "tail": embeds.clone()}

    def _forward_text(self, embeds, paddings):
        model = self.text_model
        x = embeds
        if model.scale_sqrt_depth:
            x = x * (model.embedding_dim ** 0.5)
        x = x + model.pos_embedder(seq_length=x.shape[1]).to(x.device)
        mask = (paddings == 0).float().permute(1, 0)
        x = model.transformer(x.permute(1, 0, 2), mask).permute(1, 0, 2)
        x = model.ln_final(x)
        return model.pooling(x, compatible_paddings=paddings[:, :, None])

    def encode_text(self, ctx, aux):
        return self._forward_text(torch.cat([ctx, aux["tail"]], dim=1),
                                  aux["paddings"])

    @torch.no_grad()
    def encode_fixed_text(self, texts):
        ids, paddings = self._tokenize(texts)
        return self._forward_text(self.text_model.token_embedding(ids), paddings)


# Register TIPS factories into the shared BACKBONES registry (created in setup).
if globals().get("TIPS_SOURCE_AVAILABLE"):
    register_backbone("tips", lambda: TipsBackbone(version="v1", name="tips"))
    register_backbone("tips_v2", lambda: TipsBackbone(version="v2", name="tips_v2"))
else:
    skip_backbone("tips", "google-deepmind/tips source not available "
                  "(re-run setup cell; need git clone of tips)")
    skip_backbone("tips_v2", "google-deepmind/tips source not available "
                  "(re-run setup cell; need git clone of tips)")

## Cell 8 — CLIP backbone

The same interface, so nothing downstream knows which encoder it is holding.

CLIP-specific handling:

- **Positional-embedding interpolation.** OpenAI's ViT-L/14@336px was pretrained
  at 336px (a 24×24 grid) and cannot natively accept the 518px inputs this
  protocol uses (a 37×37 grid). The grid portion of the positional embedding is
  bicubically resized once at construction, which is the same thing AnomalyCLIP
  and every other 518px CLIP-based ZSAD method does.
- **`ln_post` and `proj` are applied to patch tokens**, not just the CLS token, to
  bring the dense features into the joint image–text space where the prompts live.
- **Weights are cast to fp32.** CLIP ships fp16, and prompt gradients are far
  better behaved in fp32; inference speed is recovered through autocast instead.
- **One global token only,** so `GLOBAL_TOKEN` has no effect here — the CLS token
  is reported as both `object` and `spatial`, which keeps the scoring code
  branch-free.
- **The values trick** is implemented against CLIP's packed QKV projection and
  its `(length, batch, dim)` layout, so the value slice has to be cut out by hand.

In [ ]:
# =============================================================================
# CELL 8 -- CLIP backbone (same interface as TIPS)
# =============================================================================
try:
    import clip
except Exception as _clip_error:  # noqa: BLE001
    clip = None
    skip_backbone("clip", _clip_error)

CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)


def _clip_value_block(block, x):
    """One CLIP residual block with attention replaced by its value path."""
    normed = block.ln_1(x)
    qkv = F.linear(normed, block.attn.in_proj_weight, block.attn.in_proj_bias)
    x = x + block.attn.out_proj(qkv.chunk(3, dim=-1)[2])
    return x + block.mlp(block.ln_2(x))


def _ensure_openai_clip_checkpoint(model_name):
    """Download OpenAI CLIP weights via our resume/proxy-safe helper.

    `clip.load` uses bare urllib and the Windows system proxy; a flaky
    127.0.0.1:10809 proxy leaves a truncated ~/.cache/clip/*.pt that then fails
    SHA256 and re-downloads forever. We fetch into WEIGHTS_DIR with checksums.
    """
    import clip.clip as clip_impl

    if model_name not in clip_impl._MODELS:
        raise ValueError(f"unknown CLIP backbone {model_name!r}; "
                         f"available={list(clip_impl._MODELS)}")
    url = clip_impl._MODELS[model_name]
    expected_sha = url.split("/")[-2]
    dest_dir = os.path.join(WEIGHTS_DIR, "clip")
    dest = os.path.join(dest_dir, os.path.basename(url))
    # Drop the known-bad default-cache copy so a fallback path cannot reuse it.
    bad_default = os.path.join(os.path.expanduser("~/.cache/clip"),
                               os.path.basename(url))
    if os.path.isfile(bad_default):
        try:
            if _file_sha256(bad_default) != expected_sha:
                os.remove(bad_default)
                print(f"  removed corrupt CLIP cache {bad_default}", flush=True)
        except OSError:
            pass
    return _download(url, dest, retries=4, sha256=expected_sha)


class ClipBackbone:
    name = "clip"
    has_two_global_tokens = False

    def __init__(self, device=None):
        self.device = device or DEVICE
        ckpt = _ensure_openai_clip_checkpoint(CLIP_BACKBONE)
        # Checkpoint is already local; load is offline. Keep proxy fallback in
        # case clip.load still probes the network for a JIT/metadata path.
        model, _ = run_with_proxy_fallback(
            lambda: clip.load(ckpt, device=self.device, jit=False),
            label="clip.load")
        self.model = model.float().eval()
        for parameter in self.model.parameters():
            parameter.requires_grad_(False)

        self.visual = self.model.visual
        self.patch_size = self.visual.conv1.kernel_size[0]
        self.image_size = INPUT_SIZE
        self.grid = self.image_size // self.patch_size
        self.embed_dim = self.model.text_projection.shape[1]
        self.depth = len(self.visual.transformer.resblocks)
        self.layers = resolve_dense_layers(self.name, self.depth)
        self.temperature = float(1.0 / self.model.logit_scale.exp().item())
        self.num_params = sum(p.numel() for p in self.model.parameters())

        self._pos_embed = self._interpolated_pos_embed()
        self._mean = torch.tensor(CLIP_MEAN, device=self.device).view(1, 3, 1, 1)
        self._std = torch.tensor(CLIP_STD, device=self.device).view(1, 3, 1, 1)

    def _interpolated_pos_embed(self):
        """Resizes the visual positional embedding to this notebook's input size."""
        pos = self.visual.positional_embedding.detach()
        class_pos, patch_pos = pos[:1], pos[1:]
        native = int(round(patch_pos.shape[0] ** 0.5))
        if native == self.grid:
            return pos
        patch_pos = patch_pos.reshape(1, native, native, -1).permute(0, 3, 1, 2)
        patch_pos = F.interpolate(patch_pos, size=(self.grid, self.grid),
                                  mode="bicubic", align_corners=False)
        patch_pos = patch_pos.permute(0, 2, 3, 1).reshape(self.grid ** 2, -1)
        return torch.cat([class_pos, patch_pos], dim=0)

    # --- vision --------------------------------------------------------------
    def preprocess(self, images_uint8):
        x = images_uint8.to(self.device, non_blocking=True).float().div_(255.0)
        if x.shape[-1] != self.image_size:
            x = F.interpolate(x, size=(self.image_size, self.image_size),
                              mode="bicubic", align_corners=False, antialias=True)
        return (x.clamp_(0.0, 1.0) - self._mean) / self._std

    def _project(self, tokens):
        """Into the joint image-text space, where the prompt embeddings live."""
        return self.visual.ln_post(tokens) @ self.visual.proj

    def encode(self, x):
        visual = self.visual
        grid = x.shape[-1] // self.patch_size
        value_layer = max(self.layers)

        tokens = visual.conv1(x)
        tokens = tokens.reshape(tokens.shape[0], tokens.shape[1], -1).permute(0, 2, 1)
        class_token = (visual.class_embedding.to(tokens.dtype)
                       .view(1, 1, -1).expand(tokens.shape[0], 1, -1))
        tokens = torch.cat([class_token, tokens], dim=1) + self._pos_embed
        hidden = visual.ln_pre(tokens).permute(1, 0, 2)      # NLD -> LND

        pre_value, dense = None, {}
        for depth, block in enumerate(visual.transformer.resblocks, start=1):
            if depth == value_layer:
                pre_value = hidden
            hidden = block(hidden)
            if depth in self.layers:
                dense[depth] = self._project(hidden.permute(1, 0, 2)[:, 1:])
        global_embedding = self._project(hidden.permute(1, 0, 2)[:, :1])[:, 0]

        if USE_VALUE_ATTENTION and pre_value is not None:
            patched = _clip_value_block(visual.transformer.resblocks[value_layer - 1],
                                        pre_value)
            dense[value_layer] = self._project(patched.permute(1, 0, 2)[:, 1:])

        return {
            "object": global_embedding,
            "spatial": global_embedding,   # CLIP has a single global token
            "dense": {layer: tokens.reshape(tokens.shape[0], grid, grid, -1)
                      for layer, tokens in dense.items()},
        }

    # --- text ----------------------------------------------------------------
    def init_prompt(self, suffixes):
        """[SOT][V1..Vn][suffix][EOT], with only the V tokens trainable."""
        texts = [f"{'X ' * N_CTX}{suffix}" for suffix in suffixes]
        tokenized = clip.tokenize(texts).to(self.device)
        with torch.no_grad():
            embeds = self.model.token_embedding(tokenized)
        seed_source = embeds[:, 1:1 + N_CTX]
        ctx = seed_source + 0.02 * torch.randn_like(seed_source)
        aux = {
            "prefix": embeds[:, :1].clone(),           # start-of-text
            "tail": embeds[:, 1 + N_CTX:].clone(),     # suffix, EOT, padding
            "eot_index": tokenized.argmax(dim=-1),
        }
        return ctx, aux

    def _forward_text(self, embeds, eot_index):
        model = self.model
        x = embeds + model.positional_embedding
        x = model.transformer(x.permute(1, 0, 2)).permute(1, 0, 2)
        x = model.ln_final(x)
        pooled = x[torch.arange(x.shape[0], device=x.device), eot_index]
        return pooled @ model.text_projection

    def encode_text(self, ctx, aux):
        embeds = torch.cat([aux["prefix"], ctx, aux["tail"]], dim=1)
        return self._forward_text(embeds, aux["eot_index"])

    @torch.no_grad()
    def encode_fixed_text(self, texts):
        tokenized = clip.tokenize(texts).to(self.device)
        return self._forward_text(self.model.token_embedding(tokenized),
                                  tokenized.argmax(dim=-1))


if clip is not None:
    register_backbone("clip", ClipBackbone)

## Cell 8b — SigLIP2 backbone

Same frozen-encoder interface as TIPS and CLIP. Loads Google's **SigLIP 2**
via OpenCLIP (`hf-hub:timm/...`), which is the backbone Tipsomaly evaluates in
Table 9 of [arXiv:2602.03594](https://arxiv.org/abs/2602.03594).

SigLIP2 was trained with a sigmoid image–text loss rather than a softmax
contrastive loss. This notebook still scores anomalies with a two-class softmax
over `{normal, anomalous}` prompt similarities — matching Tipsomaly's protocol —
so the comparison isolates the backbone under a shared prompt-learning recipe.

Requires `open-clip-torch >= 2.31` and `timm >= 1.0.15`. Architecture details
live in `model_configs/siglip2.txt`.

In [ ]:
# =============================================================================
# CELL 8b -- SigLIP2 backbone (OpenCLIP / timm)
# =============================================================================
try:
    import open_clip
except Exception as _open_clip_error:  # noqa: BLE001
    open_clip = None
    skip_backbone("siglip2", _open_clip_error)

SIGLIP2_MEAN = (0.5, 0.5, 0.5)
SIGLIP2_STD = (0.5, 0.5, 0.5)


def _siglip_text_modules(model):
    """Return (token_embedding, positional_embedding, transformer, ln_final, projection, attn_mask)."""
    if hasattr(model, "text") and hasattr(model.text, "transformer"):
        text = model.text
        projection = getattr(text, "text_projection", None)
        if projection is None:
            projection = getattr(text, "proj", None)
        return (text.token_embedding, text.positional_embedding, text.transformer,
                text.ln_final, projection, getattr(text, "attn_mask", None))
    projection = getattr(model, "text_projection", None)
    return (model.token_embedding, model.positional_embedding, model.transformer,
            model.ln_final, projection, getattr(model, "attn_mask", None))


def _project_text(pooled, projection):
    if projection is None:
        return pooled
    if isinstance(projection, torch.nn.Linear):
        return projection(pooled)
    return pooled @ projection


class SigLip2Backbone:
    """Frozen SigLIP2 encoder with the shared prompt-learning interface."""

    name = "siglip2"
    has_two_global_tokens = False

    def __init__(self, device=None, model_id=None):
        self.device = device or DEVICE
        self.model_id = model_id or SIGLIP2_MODEL
        self.image_size = INPUT_SIZE

        # open_clip >= 2.24 returns (model, preprocess) when return_transform=True
        # (the default). Older tutorials unpacked three values; that raises here.
        # HF Hub honours the Windows system proxy; on SSL EOF from a flaky
        # 127.0.0.1:10809 we retry once with proxies disabled.
        cache_dir = os.path.join(WEIGHTS_DIR, "open_clip")
        os.makedirs(cache_dir, exist_ok=True)

        def _load_siglip():
            try:
                return open_clip.create_model_from_pretrained(
                    self.model_id, cache_dir=cache_dir)
            except TypeError:
                return open_clip.create_model_from_pretrained(self.model_id)

        loaded = run_with_proxy_fallback(_load_siglip, label="siglip2/hf")
        model = loaded[0] if isinstance(loaded, (tuple, list)) else loaded
        self.tokenizer = run_with_proxy_fallback(
            lambda: open_clip.get_tokenizer(self.model_id),
            label="siglip2/tokenizer")
        self.model = model.eval().to(self.device)
        for parameter in self.model.parameters():
            parameter.requires_grad_(False)

        self.visual = self.model.visual
        self.patch_size = self._infer_patch_size()
        self.grid = self.image_size // self.patch_size
        (self._tok_emb, self._pos_emb, self._text_tower, self._ln_final,
         self._text_proj, self._attn_mask) = _siglip_text_modules(self.model)
        self.embed_dim = self._infer_embed_dim()
        self.depth = self._infer_depth()
        self.layers = resolve_dense_layers(self.name, self.depth)
        scale = self.model.logit_scale.exp().item()
        self.temperature = float(1.0 / scale)
        self.num_params = sum(p.numel() for p in self.model.parameters())
        self._mean = torch.tensor(SIGLIP2_MEAN, device=self.device).view(1, 3, 1, 1)
        self._std = torch.tensor(SIGLIP2_STD, device=self.device).view(1, 3, 1, 1)

    def _infer_embed_dim(self):
        proj = self._text_proj
        if isinstance(proj, torch.nn.Linear):
            return int(proj.out_features)
        if isinstance(proj, torch.Tensor):
            return int(proj.shape[1])
        if hasattr(self.model, "text_projection"):
            proj = self.model.text_projection
            return int(proj.out_features if isinstance(proj, torch.nn.Linear) else proj.shape[1])
        if hasattr(self.model, "embed_dim"):
            return int(self.model.embed_dim)
        return 1024

    def _infer_patch_size(self):
        visual = self.visual
        if hasattr(visual, "trunk") and hasattr(visual.trunk, "patch_embed"):
            patch = visual.trunk.patch_embed.patch_size
            return int(patch[0] if isinstance(patch, (tuple, list)) else patch)
        if hasattr(visual, "conv1"):
            return int(visual.conv1.kernel_size[0])
        return 16

    def _infer_depth(self):
        visual = self.visual
        if hasattr(visual, "trunk") and hasattr(visual.trunk, "blocks"):
            return len(visual.trunk.blocks)
        if hasattr(visual, "transformer") and hasattr(visual.transformer, "resblocks"):
            return len(visual.transformer.resblocks)
        return 24

    def preprocess(self, images_uint8):
        x = images_uint8.to(self.device, non_blocking=True).float().div_(255.0)
        if x.shape[-1] != self.image_size:
            x = F.interpolate(x, size=(self.image_size, self.image_size),
                              mode="bicubic", align_corners=False, antialias=True)
        return (x.clamp_(0.0, 1.0) - self._mean) / self._std

    def _encode_timm(self, x):
        trunk = self.visual.trunk
        tokens = trunk.patch_embed(x)
        if hasattr(trunk, "_pos_embed"):
            tokens = trunk._pos_embed(tokens)
        elif hasattr(trunk, "pos_embed"):
            tokens = tokens + trunk.pos_embed
        if hasattr(trunk, "pos_drop"):
            tokens = trunk.pos_drop(tokens)
        prefix = int(getattr(trunk, "num_prefix_tokens", 1))
        value_layer = max(self.layers)
        dense = {}
        for depth, block in enumerate(trunk.blocks, start=1):
            tokens = block(tokens)
            if depth in self.layers:
                dense[depth] = tokens[:, prefix:]
        if hasattr(trunk, "norm"):
            tokens = trunk.norm(tokens)
        global_embedding = self.model.encode_image(x)
        if global_embedding.ndim > 2:
            global_embedding = global_embedding.mean(dim=1)
        head = getattr(self.visual, "head", None)
        projected = {}
        grid = self.grid
        for layer, patches in dense.items():
            if isinstance(head, torch.nn.Linear):
                projected[layer] = head(patches)
            elif hasattr(self.visual, "proj") and self.visual.proj is not None:
                proj = self.visual.proj
                projected[layer] = (proj(patches) if isinstance(proj, torch.nn.Linear)
                                    else patches @ proj)
            else:
                projected[layer] = patches
            grid = int(round(patches.shape[1] ** 0.5))
        return {
            "object": global_embedding,
            "spatial": global_embedding,
            "dense": {layer: tokens.reshape(tokens.shape[0], grid, grid, -1)
                      for layer, tokens in projected.items()},
            "_value_layer": value_layer,
        }

    def _encode_openclip_vit(self, x):
        visual = self.visual
        grid = x.shape[-1] // self.patch_size
        tokens = visual.conv1(x)
        tokens = tokens.reshape(tokens.shape[0], tokens.shape[1], -1).permute(0, 2, 1)
        class_token = visual.class_embedding.to(tokens.dtype).view(1, 1, -1).expand(
            tokens.shape[0], 1, -1)
        tokens = torch.cat([class_token, tokens], dim=1)
        if tokens.shape[1] != visual.positional_embedding.shape[0]:
            # Interpolate positional embeddings for non-native resolutions.
            pos = visual.positional_embedding
            cls_pos, patch_pos = pos[:1], pos[1:]
            native = int(round(patch_pos.shape[0] ** 0.5))
            patch_pos = patch_pos.reshape(1, native, native, -1).permute(0, 3, 1, 2)
            patch_pos = F.interpolate(patch_pos, size=(grid, grid),
                                      mode="bicubic", align_corners=False)
            patch_pos = patch_pos.permute(0, 2, 3, 1).reshape(grid ** 2, -1)
            pos = torch.cat([cls_pos, patch_pos], dim=0)
        else:
            pos = visual.positional_embedding
        tokens = tokens + pos.to(tokens.dtype)
        tokens = visual.ln_pre(tokens).permute(1, 0, 2)
        dense = {}
        for depth, block in enumerate(visual.transformer.resblocks, start=1):
            tokens = block(tokens)
            if depth in self.layers:
                patch = tokens.permute(1, 0, 2)[:, 1:]
                if hasattr(visual, "ln_post"):
                    patch = visual.ln_post(patch)
                if getattr(visual, "proj", None) is not None:
                    patch = patch @ visual.proj
                dense[depth] = patch
        global_embedding = self.model.encode_image(x)
        return {
            "object": global_embedding,
            "spatial": global_embedding,
            "dense": {layer: tokens.reshape(tokens.shape[0], grid, grid, -1)
                      for layer, tokens in dense.items()},
        }

    def encode(self, x):
        if hasattr(self.visual, "trunk"):
            features = self._encode_timm(x)
            features.pop("_value_layer", None)
            return features
        return self._encode_openclip_vit(x)

    def init_prompt(self, suffixes):
        texts = [f"{'X ' * N_CTX}{suffix}" for suffix in suffixes]
        tokenized = self.tokenizer(texts).to(self.device)
        with torch.no_grad():
            embeds = self._tok_emb(tokenized)
        seed_source = embeds[:, 1:1 + N_CTX]
        ctx = seed_source + 0.02 * torch.randn_like(seed_source)
        aux = {
            "prefix": embeds[:, :1].clone(),
            "tail": embeds[:, 1 + N_CTX:].clone(),
            "eot_index": tokenized.argmax(dim=-1),
        }
        return ctx, aux

    def _forward_text(self, embeds, eot_index):
        x = embeds + self._pos_emb.to(embeds.dtype)
        x = x.permute(1, 0, 2)
        if self._attn_mask is not None:
            x = self._text_tower(x, attn_mask=self._attn_mask)
        else:
            x = self._text_tower(x)
        x = x.permute(1, 0, 2)
        x = self._ln_final(x)
        pooled = x[torch.arange(x.shape[0], device=x.device), eot_index]
        return _project_text(pooled, self._text_proj)

    def encode_text(self, ctx, aux):
        embeds = torch.cat([aux["prefix"], ctx, aux["tail"]], dim=1)
        return self._forward_text(embeds, aux["eot_index"])

    @torch.no_grad()
    def encode_fixed_text(self, texts):
        tokenized = self.tokenizer(texts).to(self.device)
        embeds = self._tok_emb(tokenized)
        return self._forward_text(embeds, tokenized.argmax(dim=-1))


if open_clip is not None:
    register_backbone("siglip2", SigLip2Backbone)

## Cell 8c — DINOv2.txt (dino.txt) backbone

Meta's language-aligned **DINOv2** (`dino.txt`, [arXiv:2412.16334](https://arxiv.org/abs/2412.16334)):
a frozen DINOv2 ViT-L/14 (with registers) plus a text tower trained with LiT-style
alignment. The slide deck lists this as `DINOv2.txt` next to TIPS and SigLIP2.

Global features concatenate the class token with mean-pooled patches (2048-d).
Dense maps use per-patch `[CLS || patch_i]` so every location lives in that same
joint space. As with every other backbone here, **only prompt context vectors
train**.

The machine-readable architecture card is `model_configs/dinov2.txt`.
`model_configs/dinov3.txt` documents why DINOv3.txt is not wired yet.

In [ ]:
# =============================================================================
# CELL 8c -- DINOv2.txt (dino.txt) backbone
# =============================================================================
import json

_config_dir = MODEL_CONFIG_DIR if "MODEL_CONFIG_DIR" in globals() else os.path.join(
    PROJECT_ROOT if "PROJECT_ROOT" in globals() else os.getcwd(), "model_configs")
DINOV2_CONFIG_PATH = os.path.join(_config_dir, "dinov2.txt")
if not os.path.isfile(DINOV2_CONFIG_PATH):
    raise FileNotFoundError(
        "model_configs/dinov2.txt not found; expected at "
        f"{DINOV2_CONFIG_PATH!r}. Copy the model_configs/ folder next to the notebook.")

with open(DINOV2_CONFIG_PATH, encoding="utf-8") as _fh:
    DINOV2_MODEL_CARD = json.load(_fh)

DINOV2_MEAN = tuple(DINOV2_MODEL_CARD["vision"]["mean"])
DINOV2_STD = tuple(DINOV2_MODEL_CARD["vision"]["std"])
DINOV2_HUB_ENTRY = DINOV2_MODEL_CARD["hub_entry"]


class DinoTxtBackbone:
    """Frozen dino.txt (DINOv2 + text) with the shared prompt-learning interface."""

    name = "dinov2"
    has_two_global_tokens = False

    def __init__(self, device=None, hub_repo=None, hub_entry=None):
        self.device = device or DEVICE
        self.image_size = INPUT_SIZE
        self.patch_size = int(DINOV2_MODEL_CARD["vision"]["patch_size"])
        self.grid = self.image_size // self.patch_size
        repo = hub_repo or DINOV2_MODEL_CARD["hub_repo"]
        entry = hub_entry or DINOV2_HUB_ENTRY

        # torch.hub uses urllib + the Windows system proxy. Prefetch the published
        # weight URLs with our resume/proxy-safe downloader into TORCH_HOME, then
        # load the hub entry with proxies disabled.
        hub_dir = os.path.join(WEIGHTS_DIR, "torch", "hub")
        os.makedirs(hub_dir, exist_ok=True)
        torch.hub.set_dir(hub_dir)
        for _url in DINOV2_MODEL_CARD.get("checkpoints", {}).values():
            if not isinstance(_url, str) or not _url.startswith("http"):
                continue
            _dest = os.path.join(hub_dir, "checkpoints", os.path.basename(_url))
            try:
                _download(_url, _dest, retries=4)
            except Exception as _prefetch_error:  # noqa: BLE001
                print(f"  [warn] DINOv2 prefetch failed for "
                      f"{os.path.basename(_url)}: {_prefetch_error}", flush=True)

        self.model = run_with_proxy_fallback(
            lambda: torch.hub.load(repo, entry, trust_repo=True),
            label="dinov2/hub")
        try:
            self.tokenizer = run_with_proxy_fallback(
                lambda: torch.hub.load(repo, "get_tokenizer", trust_repo=True),
                label="dinov2/tokenizer")
        except Exception:
            from dinov2.hub.dinotxt import get_tokenizer
            self.tokenizer = get_tokenizer()
        self.model = self.model.eval().to(self.device)
        for parameter in self.model.parameters():
            parameter.requires_grad_(False)

        self.embed_dim = int(DINOV2_MODEL_CARD["joint_embed_dim"])
        self.depth = int(DINOV2_MODEL_CARD["vision"]["depth"])
        self.layers = resolve_dense_layers(self.name, self.depth)
        self.temperature = float(1.0 / self.model.logit_scale.exp().item())
        self.num_params = sum(p.numel() for p in self.model.parameters())
        self._mean = torch.tensor(DINOV2_MEAN, device=self.device).view(1, 3, 1, 1)
        self._std = torch.tensor(DINOV2_STD, device=self.device).view(1, 3, 1, 1)

        text_tower = self.model.text_model
        self._text_backbone = text_tower.backbone
        self._text_head = text_tower.head
        self._pooler = text_tower.tokens_pooler_type

    def preprocess(self, images_uint8):
        x = images_uint8.to(self.device, non_blocking=True).float().div_(255.0)
        if x.shape[-1] != self.image_size:
            x = F.interpolate(x, size=(self.image_size, self.image_size),
                              mode="bicubic", align_corners=False, antialias=True)
        return (x.clamp_(0.0, 1.0) - self._mean) / self._std

    def encode(self, x):
        class_token, patch_tokens = self.model.get_visual_class_and_patch_tokens(x)
        global_embedding = torch.cat(
            [class_token, patch_tokens.mean(dim=1)], dim=-1)
        dense_tokens = torch.cat(
            [class_token.unsqueeze(1).expand(-1, patch_tokens.shape[1], -1),
             patch_tokens], dim=-1)
        grid = int(round(patch_tokens.shape[1] ** 0.5))
        layer = max(self.layers)
        return {
            "object": global_embedding,
            "spatial": global_embedding,
            "dense": {layer: dense_tokens.reshape(
                dense_tokens.shape[0], grid, grid, -1)},
        }

    def init_prompt(self, suffixes):
        texts = [f"{'X ' * N_CTX}{suffix}" for suffix in suffixes]
        tokenized = self.tokenizer.tokenize(texts).to(self.device)
        with torch.no_grad():
            embeds = self._text_backbone.token_embedding(tokenized)
        seed_source = embeds[:, 1:1 + N_CTX]
        ctx = seed_source + 0.02 * torch.randn_like(seed_source)
        aux = {
            "prefix": embeds[:, :1].clone(),
            "tail": embeds[:, 1 + N_CTX:].clone(),
            "eot_index": tokenized.argmax(dim=-1),
        }
        return ctx, aux

    def _forward_text_embeds(self, embeds, eot_index):
        backbone = self._text_backbone
        length = embeds.shape[1]
        x = embeds + backbone.positional_embedding[:length].to(embeds.dtype)
        x = backbone.dropout(x)
        for block in backbone.blocks:
            x = block(x)
        x = backbone.ln_final(x)
        x = self._text_head(x)
        if self._pooler == "argmax":
            return x[torch.arange(x.shape[0], device=x.device), eot_index]
        if self._pooler == "first":
            return x[:, 0]
        return x[:, -1]

    def encode_text(self, ctx, aux):
        embeds = torch.cat([aux["prefix"], ctx, aux["tail"]], dim=1)
        return self._forward_text_embeds(embeds, aux["eot_index"])

    @torch.no_grad()
    def encode_fixed_text(self, texts):
        tokenized = self.tokenizer.tokenize(texts).to(self.device)
        embeds = self._text_backbone.token_embedding(tokenized)
        return self._forward_text_embeds(embeds, tokenized.argmax(dim=-1))


register_backbone("dinov2", DinoTxtBackbone)

## Cell 9 — Prompts, scoring and losses

**The only trainable tensor in this notebook is defined here.** `LearnablePrompts`
holds a single parameter of shape `[2, N_CTX, D]` — 8 context tokens for the
normal state and 8 for the anomalous one. The backbone is deliberately kept in a
plain list so it is not registered as a submodule: the saved checkpoint then
contains the context vectors and nothing else, which is the audit trail for the
claim that no internal parameter is touched.

**Fixed prompts** are a Cartesian product of generic inspection templates and
state phrases (`a cropped photo of a flawless {}` … `a photo of a {} with defect`),
encoded, averaged within each state, and re-normalised into two prototypes. They
use the real category name, which is available in the zero-shot setting; set
`FIXED_PROMPT_CLASS_NAME = "object"` for the fully category-blind variant.

**Learnable prompts** follow AnomalyCLIP's object-agnostic form — learnable
context followed by a fixed suffix, `[V1..V8] object` and `[W1..W8] damaged
object` — so they encode a state rather than a product and can transfer to the
other dataset's categories.

**Scoring.** Patch and text embeddings are compared by cosine similarity, scaled
by the backbone's own contrastive temperature, and softmaxed over
{normal, anomalous}; the anomalous channel is the anomaly map. It is upsampled to
`MAP_RES`, Gaussian-smoothed and clamped. The image score is the global token's
anomalous probability plus the strongest local evidence, `max(map)` — the global
term alone misses small defects, the map alone is noisy. Layer aggregation, where
enabled, averages logits across the selected blocks.

**Three modes from two text sets.** `fixed` and `learned` each use their own
prompts for both the map and the score; `decoupled` takes the score from the fixed
ensemble and the map from the learned context. All three come out of one visual
forward pass, so measuring all three costs almost nothing over measuring one.

**Loss.** `LOSS_MODE = "local"` — focal plus dice on the anomaly map only, which is
Tipsomaly's default and the setting their Table 2(b) finds best for localisation.
`"global"` (image-level cross-entropy) and `"both"` are available to reproduce
that ablation.

In [ ]:
# =============================================================================
# CELL 9 -- Prompts, scoring and losses
# =============================================================================
import torch.nn as nn

FIXED_TEMPLATES = (
    "a photo of a {}.",
    "a cropped photo of a {}.",
    "a close-up photo of a {}.",
    "a bright photo of a {}.",
    "a dark photo of a {}.",
    "a blurry photo of a {}.",
    "a photo of a {} for visual inspection.",
)
NORMAL_STATES = ("{}", "flawless {}", "perfect {}", "unblemished {}",
                 "{} without defect", "{} without damage")
ANOMALOUS_STATES = ("damaged {}", "flawed {}", "{} with defect",
                    "{} with damage", "{} with flaw", "broken {}")


def fixed_prompt_texts(category):
    name = FIXED_PROMPT_CLASS_NAME or prompt_class_name(category)
    return tuple(
        [template.format(state.format(name))
         for template in FIXED_TEMPLATES for state in states]
        for states in (NORMAL_STATES, ANOMALOUS_STATES))


@torch.no_grad()
def build_fixed_text(backbone, category):
    """Two prototypes: the mean unit embedding of each state's prompt ensemble."""
    prototypes = []
    for texts in fixed_prompt_texts(category):
        embeddings = F.normalize(backbone.encode_fixed_text(list(texts)).float(), dim=-1)
        prototypes.append(F.normalize(embeddings.mean(dim=0), dim=-1))
    return torch.stack(prototypes)                       # [2, D]


class LearnablePrompts(nn.Module):
    """The notebook's only trainable parameters: 2 x N_CTX context vectors."""

    def __init__(self, backbone):
        super().__init__()
        suffixes = [LEARNABLE_SUFFIX["normal"], LEARNABLE_SUFFIX["anomalous"]]
        ctx, aux = backbone.init_prompt(suffixes)
        self.ctx = nn.Parameter(ctx.detach().float().clone())
        self._backbone = [backbone]                      # hidden from state_dict
        self._aux = aux

    def forward(self):
        embeddings = self._backbone[0].encode_text(self.ctx, self._aux)
        return F.normalize(embeddings.float(), dim=-1)   # [2, D]


def dense_logits(dense, text, logit_scale):
    """Patch-text logits, averaged over the selected blocks -> [B, 2, h, w]."""
    per_layer = []
    for _, tokens in sorted(dense.items()):
        normed = F.normalize(tokens.float(), dim=-1)
        per_layer.append(logit_scale * torch.einsum("bhwd,kd->bkhw", normed, text))
    return torch.stack(per_layer).mean(dim=0)


def global_logits(features, text, logit_scale, has_two_tokens):
    token = GLOBAL_TOKEN if has_two_tokens else "object"
    normed = F.normalize(features[token].float(), dim=-1)
    return logit_scale * normed @ text.t()               # [B, 2]


def _gaussian_blur(maps, sigma=GAUSSIAN_SIGMA):
    if not sigma:
        return maps
    radius = max(1, int(round(3 * sigma)))
    grid = torch.arange(-radius, radius + 1, device=maps.device, dtype=maps.dtype)
    kernel = torch.exp(-(grid ** 2) / (2 * sigma ** 2))
    kernel = kernel / kernel.sum()
    maps = F.conv2d(maps, kernel.view(1, 1, 1, -1), padding=(0, radius))
    return F.conv2d(maps, kernel.view(1, 1, -1, 1), padding=(radius, 0))


def to_anomaly_map(logits, map_res=None):
    """Anomalous-class probability, upsampled to the stored map resolution."""
    size = map_res or MAP_RES
    probability = logits.softmax(dim=1)[:, 1:2]
    probability = F.interpolate(probability, size=(size, size), mode="bilinear",
                                align_corners=False)
    return _gaussian_blur(probability).clamp_(0.0, 1.0)[:, 0]


def peak_evidence(maps):
    """The strongest local evidence: max(), or a top-k mean if configured."""
    flat = maps.flatten(1)
    if TOPK_FRACTION and TOPK_FRACTION > 0:
        k = max(1, int(round(TOPK_FRACTION * flat.shape[1])))
        return flat.topk(k, dim=1).values.mean(dim=1)
    return flat.max(dim=1).values


@torch.no_grad()
def anomaly_outputs(backbone, texts, images_uint8, map_res=None):
    """{mode: (scores [B], maps [B,M,M])} for every configured prompt mode."""
    logit_scale = 1.0 / backbone.temperature
    with torch.autocast("cuda", enabled=(AMP and DEVICE == "cuda")):
        features = backbone.encode(backbone.preprocess(images_uint8))

    maps, global_probability = {}, {}
    for key, text in texts.items():
        maps[key] = to_anomaly_map(dense_logits(features["dense"], text, logit_scale),
                                   map_res)
        global_probability[key] = global_logits(
            features, text, logit_scale, backbone.has_two_global_tokens).softmax(-1)[:, 1]

    def score(global_key, map_key):
        value = global_probability[global_key]
        return value + peak_evidence(maps[map_key]) if ADD_LOCAL_EVIDENCE else value

    pairs = {"fixed": ("fixed", "fixed"),
             "learned": ("learned", "learned"),
             "decoupled": ("fixed", "learned")}
    return {mode: (score(*pairs[mode]).float(), maps[pairs[mode][1]].float())
            for mode in PROMPT_MODES if all(k in texts for k in pairs[mode])}


def training_logits(backbone, prompts, images_uint8):
    with torch.autocast("cuda", enabled=(AMP and DEVICE == "cuda")):
        features = backbone.encode(backbone.preprocess(images_uint8))
    text = prompts()
    logit_scale = 1.0 / backbone.temperature
    return (global_logits(features, text, logit_scale, backbone.has_two_global_tokens),
            dense_logits(features["dense"], text, logit_scale))


def focal_loss(probability, target, gamma=FOCAL_GAMMA):
    """Multi-class focal loss over {normal, anomalous} at every pixel."""
    probability = probability.clamp(1e-6, 1.0 - 1e-6)
    truth = probability.gather(1, target[:, None])[:, 0]
    return -((1.0 - truth) ** gamma * truth.log()).mean()


def dice_loss(probability, target, eps=1.0):
    intersection = (probability * target).sum(dim=(1, 2))
    cardinality = probability.sum(dim=(1, 2)) + target.sum(dim=(1, 2))
    return (1.0 - (2.0 * intersection + eps) / (cardinality + eps)).mean()


def prompt_loss(image_logits, pixel_logits, labels, masks):
    """Focal + dice on the map, cross-entropy on the score, per LOSS_MODE."""
    total = image_logits.new_zeros(())
    if LOSS_MODE in ("global", "both"):
        total = total + IMAGE_LOSS_WEIGHT * F.cross_entropy(image_logits, labels)
    if LOSS_MODE in ("local", "both"):
        target = F.interpolate(masks[:, None].float(),
                               size=pixel_logits.shape[-2:], mode="area") > 0.5
        target = target[:, 0].long()
        probability = pixel_logits.softmax(dim=1)
        pixel = (focal_loss(probability, target)
                 + dice_loss(probability[:, 1], target.float())
                 + dice_loss(probability[:, 0], 1.0 - target.float()))
        total = total + PIXEL_LOSS_WEIGHT * pixel
    return total

## Cell 10 — Fitting the prompts

Trains one set of context vectors per (backbone, source dataset): two runs per
backbone, one on MVTec's test split and one on VisA's, each used only to evaluate
the *other* dataset.

Hyperparameters follow Tipsomaly exactly — Adam, lr 1e-3, betas (0.5, 0.999),
2 epochs, batch size 8, seed 111 — with cosine decay and gradient clipping. Two
epochs sounds short, but there are only 16 vectors to fit and their Figure 3
shows longer prompts and longer schedules overfitting the source domain.

The categories of the source dataset are concatenated and shuffled together, so a
batch mixes products and the context cannot specialise to one of them. Training
images are clean: corruption is an evaluation-time variable only, so the numbers
in the report measure robustness to perturbations the prompts have never seen.

Checkpoints hold the context tensor alone, and are keyed by backbone, source
dataset, loss mode and seed. The function returns early if one exists, so a
restarted session does not retrain — and the assertion on the parameter count is
the mechanical check that nothing but prompts is being optimised.

In [ ]:
# =============================================================================
# CELL 10 -- Fitting the prompt context
# =============================================================================
import time


def prompt_checkpoint_path(model_name, source):
    return os.path.join(
        CHECKPOINT_DIR,
        f"{model_name}_{source}_{LOSS_MODE}_ctx{N_CTX}_seed{SEED}.pt")


def build_train_loader(source):
    parts = [AnomalyTestSet(source, category, limit=MAX_TRAIN_IMAGES_PER_CATEGORY)
             for category in CATEGORIES[source]]
    return DataLoader(
        ConcatDataset(parts), batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"), drop_last=False,
        **dataloader_kwargs(derived_seed("train", source)))


def assert_prompt_learning_only(backbone, prompts):
    """Pptx constraint: encoders frozen; only prompt context vectors train."""
    for module_name in ("image_model", "text_model", "model", "visual"):
        module = getattr(backbone, module_name, None)
        if module is None or not isinstance(module, torch.nn.Module):
            continue
        for param in module.parameters():
            if param.requires_grad:
                raise AssertionError(
                    f"pptx protocol violated: {module_name} has trainable "
                    "encoder parameters (prompt-learning only allowed)")
    trainable = [p for p in prompts.parameters() if p.requires_grad]
    assert len(trainable) == 1 and trainable[0].numel() == 2 * N_CTX * backbone.embed_dim, \
        "only the prompt context may be trainable"
    assert set(PPTX_REQUIRED_PROMPT_MODES) <= set(PROMPT_MODES), \
        "pptx slide 23 requires fixed and learned prompt modes"
    return trainable


def train_prompts(backbone, source, verbose=True):
    """Fits the context vectors on `source`'s test split. Resumes if checkpointed."""
    path = prompt_checkpoint_path(backbone.name, source)
    prompts = LearnablePrompts(backbone).to(DEVICE)
    trainable = assert_prompt_learning_only(backbone, prompts)

    if RESUME and os.path.isfile(path):
        prompts.load_state_dict(torch.load(path, map_location=DEVICE))
        if verbose:
            print(f"  [{backbone.name}/{source}] loaded {os.path.basename(path)}")
        return prompts.eval()

    seed_everything(derived_seed("train", backbone.name, source))
    loader = build_train_loader(source)
    optimizer = torch.optim.Adam(trainable, lr=LR, betas=ADAM_BETAS,
                                 weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, EPOCHS * len(loader)))

    if verbose:
        print(f"  [{backbone.name}/{source}] fitting {trainable[0].numel():,} "
              f"prompt parameters on {len(loader.dataset)} images "
              f"({EPOCHS} epochs, {LOSS_MODE} loss)")
    prompts.train()
    for epoch in range(EPOCHS):
        started, running, seen = time.time(), 0.0, 0
        for batch in loader:
            image_logits, pixel_logits = training_logits(backbone, prompts,
                                                         batch["image"])
            loss = prompt_loss(image_logits, pixel_logits,
                               batch["label"].to(DEVICE),
                               batch["mask"].to(DEVICE))
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            running += loss.item() * batch["label"].shape[0]
            seen += batch["label"].shape[0]
        if verbose:
            print(f"    epoch {epoch + 1}/{EPOCHS}  loss {running / max(seen, 1):.4f}"
                  f"  ({time.time() - started:.0f}s)")

    prompts.eval()
    torch.save(prompts.state_dict(), path)
    return prompts

## Cell 11 — Inference and artefact storage

One *shard* is one (model, dataset, category, corruption, severity) cell. A shard
runs the encoder once over the category's test images and writes one `.npz` per
prompt mode containing:

- `maps` — float16 `[N, 64, 64]` anomaly maps, the low-resolution store
- `scores` — float32 `[N]` image-level anomaly scores
- `labels` — the image-level ground truth
- metadata: backbone, dense layers, temperature, resolutions, seed, config hash

float16 at 64×64 is ~8 KB per image; the full sweep is a few gigabytes rather
than the hundreds that full-resolution maps would need. The precision loss is
irrelevant to ranking metrics, which only use the ordering of scores.

Ground truth is written separately, once per category — except for geometric
corruptions, where the warp moves the defect and each (corruption, severity) needs
its own masks. `ground_truth_path` encodes that distinction, which is what keeps
storage flat while staying correct.

`RESUME` skips any shard whose file already exists, so an interrupted sweep is
restarted by simply re-running the cell. Because corruption seeds are derived from
image identity rather than a global stream, the resumed run produces bit-identical
data to an uninterrupted one.

In [ ]:
# =============================================================================
# CELL 11 -- Inference and artefact storage
# =============================================================================
import json


def config_fingerprint():
    """Identifies the settings a stored artefact was produced under."""
    payload = {
        "seed": SEED, "input": INPUT_SIZE, "map_res": MAP_RES,
        "models": list(MODELS),
        "tips": f"{TIPS_VERSION}-{TIPS_VARIANT}", "clip": CLIP_BACKBONE,
        "siglip2": SIGLIP2_MODEL, "dinov2": "model_configs/dinov2.txt",
        "layers": DENSE_LAYER_FRACTIONS if SHARED_DENSE_LAYERS is None
        else {"shared": SHARED_DENSE_LAYERS},
        "value_attention": USE_VALUE_ATTENTION, "n_ctx": N_CTX,
        "loss": LOSS_MODE, "global_token": GLOBAL_TOKEN,
        "local_evidence": ADD_LOCAL_EVIDENCE, "sigma": GAUSSIAN_SIGMA,
    }
    blob = json.dumps(payload, sort_keys=True, default=str)
    return hashlib.blake2b(blob.encode(), digest_size=6).hexdigest()


CONFIG_ID = config_fingerprint()


def shard_path(model_name, prompt_mode, dataset, category, corruption, severity):
    directory = os.path.join(ARTIFACT_DIR, model_name, prompt_mode, dataset, category)
    os.makedirs(directory, exist_ok=True)
    return os.path.join(directory, f"{corruption}_s{severity}.npz")


def ground_truth_path(dataset, category, corruption, severity):
    """Geometric warps move the defect, so those masks are stored per setting."""
    directory = os.path.join(ARTIFACT_DIR, "_ground_truth", dataset, category)
    os.makedirs(directory, exist_ok=True)
    stem = (f"{corruption}_s{severity}" if corruption in GEOMETRIC else "base")
    return os.path.join(directory, f"{stem}.npz")


def save_ground_truth(dataset, category, corruption, severity, masks, labels):
    path = ground_truth_path(dataset, category, corruption, severity)
    if os.path.isfile(path):
        return path
    np.savez_compressed(path, masks=np.packbits(masks, axis=-1),
                        labels=labels.astype(np.uint8),
                        shape=np.array(masks.shape, dtype=np.int32))
    return path


def load_ground_truth(dataset, category, corruption="clean", severity=0):
    stored = np.load(ground_truth_path(dataset, category, corruption, severity))
    shape = tuple(int(value) for value in stored["shape"])
    masks = np.unpackbits(stored["masks"], axis=-1, count=shape[-1])
    return masks.reshape(shape).astype(np.uint8), stored["labels"].astype(np.int64)


def run_shard(backbone, texts, dataset, category, corruption, severity,
              limit=None, save=True):
    """Scores one (model, dataset, category, corruption, severity) cell."""
    loader = make_loader(dataset, category, corruption, severity, limit=limit)
    collected = {}
    masks, labels = [], []

    for batch in loader:
        outputs = anomaly_outputs(backbone, texts, batch["image"])
        for mode, (scores, maps) in outputs.items():
            store = collected.setdefault(mode, {"scores": [], "maps": []})
            store["scores"].append(scores.cpu().numpy().astype(np.float32))
            store["maps"].append(maps.cpu().numpy().astype(np.float16))
        masks.append(batch["mask"].numpy().astype(np.uint8))
        labels.append(batch["label"].numpy().astype(np.int64))

    masks = np.concatenate(masks)
    labels = np.concatenate(labels)
    results = {}
    for mode, store in collected.items():
        results[mode] = {
            "scores": np.concatenate(store["scores"]),
            "maps": np.concatenate(store["maps"]),
            "labels": labels,
        }
    if not save:
        return results, masks

    save_ground_truth(dataset, category, corruption, severity, masks, labels)
    for mode, payload in results.items():
        np.savez_compressed(
            shard_path(backbone.name, mode, dataset, category, corruption, severity),
            scores=payload["scores"], maps=payload["maps"], labels=labels,
            meta=json.dumps({
                "model": backbone.name, "prompt_mode": mode,
                "dataset": dataset, "category": category,
                "corruption": corruption, "severity": int(severity),
                "dense_layers": list(backbone.layers), "depth": backbone.depth,
                "temperature": backbone.temperature, "input_size": INPUT_SIZE,
                "map_res": MAP_RES, "seed": SEED, "config_id": CONFIG_ID,
            }))
    return results, masks


def load_shard(model_name, prompt_mode, dataset, category, corruption, severity):
    path = shard_path(model_name, prompt_mode, dataset, category, corruption, severity)
    if not os.path.isfile(path):
        return None
    stored = np.load(path, allow_pickle=False)
    return {"scores": stored["scores"].astype(np.float64),
            "maps": stored["maps"].astype(np.float32),
            "labels": stored["labels"].astype(np.int64),
            "meta": json.loads(str(stored["meta"]))}


def shard_is_done(model_name, dataset, category, corruption, severity):
    return all(os.path.isfile(shard_path(model_name, mode, dataset, category,
                                         corruption, severity))
               for mode in PROMPT_MODES)

## Cell 12 — Metrics

The eight requested numbers, computed from the stored artefacts:

| Level | Metrics |
| --- | --- |
| Pixel | AUROC, F1-max, AUPRO, optimum threshold |
| Image | AUROC, F1-max, AP, optimum threshold |

Notes on the ones that are easy to get subtly wrong:

- **Optimum threshold** is the score that maximises F1, read off the same
  precision–recall sweep that produces F1-max. It is reported on the raw score
  scale the maps are stored on, *not* rescaled to a percentage, since its purpose
  is to be usable as an operating point.
- **AUPRO** is per-connected-region overlap averaged over regions — a large defect
  and a tiny one count equally, which is the whole point of the metric — plotted
  against false-positive rate and integrated up to FPR 0.3, then normalised by
  0.3. Regions come from `skimage.measure.label` per image. Implemented over a
  fixed 200-point threshold grid so cost does not grow with the number of pixels.
- **A category with no anomalous pixel** (or no anomalous image) yields `nan`
  rather than 0, and `nan` is skipped when averaging. Scoring an undefined metric
  as zero would silently drag dataset-level means down.
- **Pixel metrics are computed at the stored map resolution.** Everything is
  therefore reproducible from the artefacts; the published-comparison cell
  re-scores the clean setting at higher resolution where absolute comparability
  matters.

In [ ]:
# =============================================================================
# CELL 12 -- Metrics
# =============================================================================
from skimage import measure
from sklearn.metrics import auc, average_precision_score, precision_recall_curve
from sklearn.metrics import roc_auc_score, roc_curve

_trapezoid = getattr(np, "trapezoid", None) or np.trapz

PIXEL_METRICS = ("pixel_auroc", "pixel_f1max", "pixel_aupro", "pixel_threshold")
IMAGE_METRICS = ("image_auroc", "image_f1max", "image_ap", "image_threshold")
# Slide 24 calibration viewpoint (ECE). Reported separately so ranking tables
# that expect the six classic metrics stay unchanged.
CALIBRATION_METRICS = ("image_ece",)


def _f1_max_and_threshold(truth, score):
    """F1-max and the threshold that attains it."""
    precision, recall, thresholds = precision_recall_curve(truth, score)
    denominator = precision + recall
    f1 = np.divide(2 * precision * recall, denominator,
                   out=np.zeros_like(denominator), where=denominator > 0)
    best = int(np.argmax(f1[:-1])) if len(f1) > 1 else 0
    threshold = float(thresholds[best]) if len(thresholds) else float("nan")
    return float(f1[best]), threshold


def expected_calibration_error(labels, scores, n_bins=15):
    """Binary ECE (Guo et al., ICML 2017) — pptx slide 24 calibration plot."""
    labels = np.asarray(labels).astype(int).ravel()
    scores = np.asarray(scores, dtype=np.float64).ravel()
    if labels.size == 0 or labels.min() == labels.max():
        return float("nan")
    # Map raw anomaly scores to [0, 1] probabilities via rank-preserving min-max.
    low, high = float(scores.min()), float(scores.max())
    if high <= low:
        probs = np.full_like(scores, 0.5)
    else:
        probs = (scores - low) / (high - low)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    total = labels.size
    ece = 0.0
    for left, right in zip(edges[:-1], edges[1:]):
        mask = (probs >= left) & (probs < right if right < 1.0 else probs <= right)
        if not mask.any():
            continue
        acc = float(labels[mask].mean())
        conf = float(probs[mask].mean())
        ece += (mask.sum() / total) * abs(acc - conf)
    return float(ece)


def image_metrics(labels, scores):
    labels = np.asarray(labels).astype(int).ravel()
    scores = np.asarray(scores, dtype=np.float64).ravel()
    if labels.min() == labels.max():          # single-class: undefined, not zero
        return dict.fromkeys(IMAGE_METRICS, float("nan"))
    f1, threshold = _f1_max_and_threshold(labels, scores)
    return {"image_auroc": float(roc_auc_score(labels, scores)),
            "image_f1max": f1,
            "image_ap": float(average_precision_score(labels, scores)),
            "image_threshold": threshold}


def compute_aupro(masks, maps, max_fpr=0.30, num_thresholds=200):
    """Region-averaged overlap vs. FPR, integrated to `max_fpr` and normalised."""
    masks = np.asarray(masks).astype(bool)
    if not masks.any() or masks.all():
        return float("nan")

    regions = []
    for index in range(masks.shape[0]):
        if not masks[index].any():
            continue
        labelled = measure.label(masks[index], connectivity=2)
        for region in measure.regionprops(labelled):
            coordinates = tuple(region.coords.T)
            regions.append((index, coordinates, float(region.area)))
    if not regions:
        return float("nan")

    normal = ~masks
    normal_total = float(normal.sum())
    lowest, highest = float(maps.min()), float(maps.max())
    if not np.isfinite(lowest) or highest <= lowest:
        return float("nan")
    thresholds = np.linspace(highest, lowest, num_thresholds)

    pro, fpr = [], []
    for threshold in thresholds:
        predicted = maps >= threshold
        pro.append(float(np.mean([predicted[index][coordinates].sum() / area
                                  for index, coordinates, area in regions])))
        fpr.append(float((predicted & normal).sum() / normal_total))

    fpr, pro = np.asarray(fpr), np.asarray(pro)
    order = np.argsort(fpr)
    fpr, pro = fpr[order], pro[order]
    if fpr[0] > 0:                                    # anchor the curve at the origin
        fpr, pro = np.concatenate([[0.0], fpr]), np.concatenate([[0.0], pro])
    # Interpolating onto a fixed grid keeps the integral well defined even when a
    # saturated map crosses max_fpr at its very first threshold -- that deserves
    # an AUPRO near zero, not a NaN.
    grid = np.linspace(0.0, max_fpr, num_thresholds)
    return float(_trapezoid(np.interp(grid, fpr, pro), grid) / max_fpr)


def pixel_metrics(masks, maps, with_aupro=True):
    masks = np.asarray(masks).astype(np.uint8)
    maps = np.asarray(maps, dtype=np.float32)
    flat_truth, flat_score = masks.ravel(), maps.ravel().astype(np.float64)
    if flat_truth.min() == flat_truth.max():
        return dict.fromkeys(PIXEL_METRICS, float("nan"))
    f1, threshold = _f1_max_and_threshold(flat_truth, flat_score)
    return {"pixel_auroc": float(roc_auc_score(flat_truth, flat_score)),
            "pixel_f1max": f1,
            "pixel_aupro": compute_aupro(masks, maps) if with_aupro else float("nan"),
            "pixel_threshold": threshold}


def evaluate(masks, maps, labels, scores, with_aupro=True, with_ece=True):
    result = {**pixel_metrics(masks, maps, with_aupro), **image_metrics(labels, scores)}
    if with_ece:
        result["image_ece"] = expected_calibration_error(labels, scores)
    return result


def resize_maps(maps, size):
    """Bilinear resampling of stored low-res maps, for higher-resolution scoring."""
    if maps.shape[-1] == size:
        return np.asarray(maps, dtype=np.float32)
    tensor = torch.from_numpy(np.asarray(maps, dtype=np.float32))[:, None]
    resized = F.interpolate(tensor, size=(size, size), mode="bilinear",
                            align_corners=False)
    return resized[:, 0].numpy()

## Cell 13 — The sweep

Builds the plan and executes it. The plan is the product

```
2 backbones × 2 protocol directions × categories × 34 (corruption, severity) cells
```

which for the default configuration is 2 × 27 × 34 = **1,836 shards** covering
about 132,000 image encodings per backbone, each scored under three prompt modes.

The loop is ordered backbone-outermost so each set of weights is loaded once. Per
backbone it fits the two prompt sets, then walks the categories, computing the
fixed-prompt prototypes once per category (they depend on the category name)
before iterating the corruption grid.

`RESUME` makes the cell idempotent: finished shards are counted and skipped, so
re-running after a disconnect continues rather than restarts. It also prints a
throughput estimate as it goes, and `load_backbones` reports each encoder's depth,
selected dense layers, temperature and parameter count — the numbers to quote when
someone asks whether the two backbones were matched fairly.

In [ ]:
# =============================================================================
# CELL 13 -- The sweep
# =============================================================================
EVAL_SOURCE = {evaluate_on: source for source, evaluate_on in PROTOCOL}


def load_backbones(names=MODELS):
    loaded = {}
    for name in names:
        if name not in BACKBONES:
            print(f"[skip] {name}: not registered ({BACKBONE_ERRORS.get(name, 'unknown')})")
            continue
        seed_everything(SEED)
        try:
            backbone = BACKBONES[name]()
        except Exception as error:  # noqa: BLE001
            skip_backbone(name, error)
            continue
        loaded[name] = backbone
        print(f"{name:<8} {backbone.embed_dim:>5}-d | {backbone.depth} blocks | "
              f"dense layers {backbone.layers} | tau {backbone.temperature:.4f} | "
              f"{backbone.num_params / 1e6:.1f}M params | {backbone.image_size}px")
    return loaded


def sweep_plan():
    plan = []
    for source, evaluate_on in PROTOCOL:
        for category in CATEGORIES[evaluate_on]:
            for corruption, severity in corruption_grid():
                plan.append({"source": source, "dataset": evaluate_on,
                             "category": category, "corruption": corruption,
                             "severity": severity})
    return plan


def run_sweep(backbones=None, plan=None, limit=None, verbose=True):
    """Pptx ZSAD/backbone sweep for the current SEED (slide 21 default: 111).

    Slide 10 asks for multi-seed reproducibility. To run extra seeds without
    clobbering artefacts, change SEED (and preferably OUT_ROOT) then re-run:
        SEED = 222; OUT_ROOT = f"/content/results_seed{SEED}"; ...
    """
    if SEED not in SEEDS:
        print(f"note: SEED={SEED} is outside SEEDS={SEEDS}; continuing anyway")
    backbones = backbones if backbones is not None else load_backbones()
    plan = plan if plan is not None else sweep_plan()
    total = len(plan) * len(backbones)
    done = executed = 0
    started = time.time()

    for name, backbone in backbones.items():
        # One prompt set per source dataset; each is used only on the other one.
        prompts = {source: train_prompts(backbone, source)
                   for source, _ in PROTOCOL}
        learned_text = {source: module().detach()
                        for source, module in prompts.items()}

        current_key, fixed_text = None, None
        for item in plan:
            done += 1
            if RESUME and limit is None and shard_is_done(
                    name, item["dataset"], item["category"],
                    item["corruption"], item["severity"]):
                continue

            key = (name, item["dataset"], item["category"])
            if key != current_key:
                fixed_text = build_fixed_text(backbone, item["category"])
                current_key = key

            texts = {"fixed": fixed_text, "learned": learned_text[item["source"]]}
            run_shard(backbone, texts, item["dataset"], item["category"],
                      item["corruption"], item["severity"], limit=limit,
                      save=(limit is None))
            executed += 1

            if verbose and executed % 20 == 0:
                elapsed = time.time() - started
                print(f"  {done}/{total} shards | {executed} computed | "
                      f"{elapsed / 60:.1f} min | "
                      f"{elapsed / executed:.1f}s per shard", flush=True)

    if verbose:
        print(f"sweep complete: {executed} shards computed, "
              f"{total - executed} already present "
              f"({(time.time() - started) / 60:.1f} min)")
    return backbones

## Cell 14 — Aggregation

Turns the stored shards into three tables.

**Category level** — one row per (model, prompt mode, dataset, category,
corruption, severity) with all eight metrics. This is the raw result and every
other table is derived from it.

**Dataset level** — the unweighted mean over that dataset's categories, which is
what AnomalyCLIP and Tipsomaly report, so it is the row that can be compared to
published numbers. Alongside it, `image_auroc_pooled` computes image-level metrics
over all of a dataset's images at once. The two answer different questions: the
macro mean asks how well the method does on a typical category, the pooled number
asks whether one threshold separates anomalies dataset-wide. Pooled figures are
usually lower, and the gap is itself informative about score calibration across
categories.

**Robustness** — each corrupted cell next to its clean reference, with the
absolute drop per metric. Reading `*_drop` is how you tell "TIPS is better" from
"TIPS is more robust", which are separate claims.

Because averaging skips `nan`, categories where a metric is undefined do not
distort a dataset row.

In [ ]:
# =============================================================================
# CELL 14 -- Aggregation into report tables
# =============================================================================
import pandas as pd

ALL_METRICS = PIXEL_METRICS + IMAGE_METRICS
KEY_COLUMNS = ("model", "prompt_mode", "dataset", "category", "corruption", "severity")


def collect_category_table(verbose=True):
    rows, missing = [], 0
    for name in MODELS:
        for mode in PROMPT_MODES:
            for source, evaluate_on in PROTOCOL:
                for category in CATEGORIES[evaluate_on]:
                    for corruption, severity in corruption_grid():
                        shard = load_shard(name, mode, evaluate_on, category,
                                           corruption, severity)
                        if shard is None:
                            missing += 1
                            continue
                        masks, _ = load_ground_truth(evaluate_on, category,
                                                     corruption, severity)
                        metrics = evaluate(masks, shard["maps"],
                                           shard["labels"], shard["scores"])
                        rows.append({"model": name, "prompt_mode": mode,
                                     "source": source, "dataset": evaluate_on,
                                     "category": category, "corruption": corruption,
                                     "severity": severity,
                                     "group": CORRUPTION_GROUP_OF.get(corruption, "clean"),
                                     "n_images": len(shard["labels"]),
                                     **metrics})
    if verbose and missing:
        print(f"note: {missing} shard(s) not yet computed and skipped")
    return pd.DataFrame(rows)


def pooled_image_metrics(model_name, prompt_mode, dataset, corruption, severity):
    """Image metrics over every image of a dataset at once, not per category."""
    labels, scores = [], []
    for category in CATEGORIES[dataset]:
        shard = load_shard(model_name, prompt_mode, dataset, category,
                           corruption, severity)
        if shard is None:
            return {}
        labels.append(shard["labels"])
        scores.append(shard["scores"])
    metrics = image_metrics(np.concatenate(labels), np.concatenate(scores))
    return {f"{key}_pooled": value for key, value in metrics.items()}


def build_dataset_table(category_table):
    """Dataset-level = unweighted mean over categories, as in AnomalyCLIP."""
    group_keys = ["model", "prompt_mode", "dataset", "corruption", "severity", "group"]
    aggregated = (category_table
                  .groupby(group_keys, dropna=False)[list(ALL_METRICS)]
                  .mean()
                  .reset_index())
    aggregated["n_categories"] = (category_table
                                  .groupby(group_keys, dropna=False)["category"]
                                  .nunique().values)
    pooled = [pooled_image_metrics(row.model, row.prompt_mode, row.dataset,
                                   row.corruption, row.severity)
              for row in aggregated.itertuples()]
    return pd.concat([aggregated, pd.DataFrame(pooled)], axis=1)


def build_robustness_table(dataset_table):
    """Each corrupted cell beside its clean reference, plus the absolute drop."""
    clean = (dataset_table[dataset_table["corruption"] == "clean"]
             .set_index(["model", "prompt_mode", "dataset"])[list(ALL_METRICS)])
    corrupted = dataset_table[dataset_table["corruption"] != "clean"].copy()
    if clean.empty or corrupted.empty:
        return corrupted
    index = pd.MultiIndex.from_frame(
        corrupted[["model", "prompt_mode", "dataset"]])
    for metric in ALL_METRICS:
        reference = clean[metric].reindex(index).to_numpy()
        corrupted[f"{metric}_clean"] = reference
        corrupted[f"{metric}_drop"] = reference - corrupted[metric].to_numpy()
    return corrupted


def save_tables(tables):
    for stem, table in tables.items():
        path = os.path.join(TABLE_DIR, f"{stem}_{CONFIG_ID}.csv")
        table.to_csv(path, index=False)
        print(f"wrote {path}  ({len(table)} rows)")

## Cell 15 — Smoke test (correctness + pptx complexity snapshot)

Before the full sweep, encode a few images under clean / noise / geometric
settings and verify maps, scores and metrics look sane.

Also records a **complexity / calibration** snapshot (pptx slides 10 & 24):
frozen parameter count, trainable prompt count, ms/image, peak CUDA memory, and
image-level ECE. Written to `COMPLEXITY_DIR`.

Asserts the pptx constraint: encoders frozen, only prompt context trains, and
`fixed` + `learned` prompt modes are enabled.

In [ ]:
# =============================================================================
# CELL 15 -- Smoke test (+ pptx complexity / calibration snapshot)
# =============================================================================
SMOKE_IMAGES = 8


def _peak_cuda_mb():
    if DEVICE != "cuda" or not torch.cuda.is_available():
        return float("nan")
    return float(torch.cuda.max_memory_allocated() / (1024 ** 2))


def measure_complexity(backbone, texts, dataset, category, images=SMOKE_IMAGES):
    """Slide 10/24 complexity: params, ms/img, peak VRAM (pptx Calibration/Complexity)."""
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    started = time.time()
    results, masks = run_shard(backbone, texts, dataset, category,
                               "clean", 0, limit=images, save=False)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    elapsed = time.time() - started
    prompts = LearnablePrompts(backbone)
    trainable = sum(p.numel() for p in prompts.parameters() if p.requires_grad)
    mode = "decoupled" if "decoupled" in results else next(iter(results))
    metrics = evaluate(masks, results[mode]["maps"], results[mode]["labels"],
                       results[mode]["scores"])
    return {
        "model": backbone.name,
        "embed_dim": backbone.embed_dim,
        "depth": backbone.depth,
        "frozen_params_M": backbone.num_params / 1e6,
        "trainable_params": trainable,
        "ms_per_image": 1000.0 * elapsed / max(images, 1),
        "peak_cuda_mb": _peak_cuda_mb(),
        "image_auroc": metrics["image_auroc"],
        "pixel_auroc": metrics["pixel_auroc"],
        "image_ece": metrics.get("image_ece", float("nan")),
    }


def smoke_test(images=SMOKE_IMAGES):
    seed_everything(SEED)
    available = [name for name in MODELS if name in BACKBONES]
    missing = [name for name in MODELS if name not in BACKBONES]
    if missing:
        print(f"[warn] skipping unavailable MODELS {missing}; "
              f"registered={sorted(BACKBONES)}; errors={BACKBONE_ERRORS}")
    if not available:
        print("[warn] no backbones registered -- smoke test skipped "
              "(install deps / clone tips, then re-run backbone cells)")
        return {}
    # Need at least one indexed category before loading multi-GB weights.
    _ready = globals().get("_DATA_READY_FOR_SMOKE", globals().get("_DATA_READY", True))
    if not _ready:
        print("[warn] datasets not ready -- smoke test skipped "
              "(fix DATA_ROOT / DOWNLOAD_IF_MISSING, then re-run data + smoke)")
        return {}
    try:
        _probe_ds, _probe_cat = PROTOCOL[0][1], CATEGORIES[PROTOCOL[0][1]][0]
        index_test_split(_probe_ds, _probe_cat)
    except Exception as _probe_error:  # noqa: BLE001
        print(f"[warn] cannot index {_probe_ds}/{_probe_cat}: {_probe_error}\n"
              "  smoke test skipped until datasets are available")
        return {}
    # Pptx slide 23 required prompt modes must be active.
    assert set(PPTX_REQUIRED_PROMPT_MODES) <= set(PROMPT_MODES)

    backbones = load_backbones(available)
    if not backbones:
        print("[warn] every backbone failed to construct -- smoke skipped")
        return {}
    dataset, category = PROTOCOL[0][1], CATEGORIES[PROTOCOL[0][1]][0]
    per_shard = {}
    complexity_rows = []

    for name, backbone in backbones.items():
        prompts = LearnablePrompts(backbone).to(DEVICE)
        assert_prompt_learning_only(backbone, prompts)
        trainable = sum(p.numel() for p in prompts.parameters() if p.requires_grad)
        frozen = backbone.num_params
        print(f"\n{name}: trainable parameters {trainable:,} "
              f"({100 * trainable / (trainable + frozen):.4f}% of "
              f"{(trainable + frozen) / 1e6:.0f}M) -- prompt context only")

        texts = {"fixed": build_fixed_text(backbone, category),
                 "learned": prompts().detach()}
        complexity_rows.append(measure_complexity(backbone, texts, dataset,
                                                  category, images=images))
        for corruption, severity in [("clean", 0), ("gaussian_noise", 3),
                                     ("rotation", 3)]:
            started = time.time()
            results, masks = run_shard(backbone, texts, dataset, category,
                                       corruption, severity, limit=images,
                                       save=False)
            elapsed = time.time() - started
            per_shard[name] = elapsed / images
            for mode, payload in results.items():
                metrics = evaluate(masks, payload["maps"], payload["labels"],
                                   payload["scores"])
                print(f"  {corruption:<15} s{severity} {mode:<10} "
                      f"maps {payload['maps'].shape} "
                      f"range [{payload['maps'].min():.3f}, {payload['maps'].max():.3f}] "
                      f"| pixel AUROC {metrics['pixel_auroc']:.3f} "
                      f"| image AUROC {metrics['image_auroc']:.3f} "
                      f"| ECE {metrics['image_ece']:.3f}"
                      f"  ({elapsed / images * 1000:.0f} ms/img)")
            assert masks.shape[-1] == MAP_RES
            for payload in results.values():
                assert payload["maps"].shape[-1] == MAP_RES
                assert np.isfinite(payload["scores"]).all()

    complexity = pd.DataFrame(complexity_rows).set_index("model")
    os.makedirs(COMPLEXITY_DIR, exist_ok=True)
    complexity_path = os.path.join(COMPLEXITY_DIR, f"smoke_complexity_seed{SEED}.csv")
    complexity.to_csv(complexity_path)
    print("\n=== Complexity / calibration snapshot (pptx slides 10 & 24) ===")
    display(complexity.round(3))
    print(f"wrote {complexity_path}")

    images_total = sum(len(index_test_split(evaluate_on, category))
                       for _, evaluate_on in PROTOCOL
                       for category in CATEGORIES[evaluate_on])
    settings = len(corruption_grid())
    estimate = sum(per_shard.values()) * images_total * settings / 3600
    print(f"\n{images_total} test images x {settings} settings x {len(backbones)} "
          f"backbones -> estimated sweep time {estimate:.1f} h "
          f"(plus {2 * len(backbones)} prompt fits)")
    print(f"pptx MODELS={MODELS}; for a cheap rehearsal set "
          f"MODELS = DEMO_MODELS {DEMO_MODELS}")
    return backbones


SMOKE_BACKBONES = smoke_test()

## Cell 16 — Execute

Fits the four prompt sets (2 backbones × 2 source datasets), runs the sweep, and
builds the tables. The encoders from the smoke test are reused so the weights are
not loaded twice.

This is the long cell — on a single mid-range GPU, expect several hours; Tipsomaly
reports its own experiments on an A6000. It is safe to interrupt: every shard is
written as it completes and `RESUME` skips what exists, so re-running the cell
after a disconnect picks up where it stopped.

The CSVs written at the end are the deliverable. They are keyed by a fingerprint
of the configuration, so tables from different settings cannot be confused with
each other.

In [ ]:
# =============================================================================
# CELL 16 -- Execute the sweep and build the tables
# =============================================================================
if not SMOKE_BACKBONES:
    print("[warn] no backbones from smoke test -- sweep skipped")
    BACKBONE_CACHE = {}
    CATEGORY_TABLE = pd.DataFrame()
    DATASET_TABLE = pd.DataFrame()
    ROBUSTNESS_TABLE = pd.DataFrame()
else:
    BACKBONE_CACHE = run_sweep(backbones=SMOKE_BACKBONES)
    CATEGORY_TABLE = collect_category_table()
    DATASET_TABLE = (build_dataset_table(CATEGORY_TABLE)
                     if not CATEGORY_TABLE.empty else pd.DataFrame())
    ROBUSTNESS_TABLE = (build_robustness_table(DATASET_TABLE)
                        if not DATASET_TABLE.empty else pd.DataFrame())
    if not CATEGORY_TABLE.empty:
        save_tables({"category_level": CATEGORY_TABLE,
                     "dataset_level": DATASET_TABLE,
                     "robustness": ROBUSTNESS_TABLE})

print(f"\nconfig {CONFIG_ID} | {len(CATEGORY_TABLE)} category rows | "
      f"{len(DATASET_TABLE)} dataset rows")

## Cell 17 — Report

The presentation layer: headline tables and figures, no new computation.

1. **Clean headline** — dataset-level metrics per backbone and prompt mode, the
   table to read first.
2. **Backbone delta** — TIPS minus CLIP for each metric, in points, so the
   comparison does not have to be done by eye.
3. **Prompt-mode comparison** — the fixed / learned / decoupled contrast, directly
   comparable to Tipsomaly's Table 2(a).
4. **Robustness** — mean degradation by corruption group and severity.
5. **Figures** — severity curves per metric, and per-corruption degradation bars.

`as_percentages` scales rates to points but leaves thresholds on their raw score
scale, since a threshold multiplied by 100 is not a threshold.

In [ ]:
# =============================================================================
# CELL 17 -- Report: headline tables and figures
# =============================================================================
import matplotlib.pyplot as plt

pd.set_option("display.width", 200, "display.max_columns", 50)

HEADLINE = ["pixel_auroc", "pixel_f1max", "pixel_aupro", "pixel_threshold",
            "image_auroc", "image_f1max", "image_ap", "image_threshold"]


def _is_rate(column):
    """Rates read better as points; thresholds must stay on the score scale."""
    # A pivot_table over several metrics labels its columns (metric, dataset).
    name = column[0] if isinstance(column, tuple) else column
    stem = name.replace("_clean", "").replace("_drop", "").replace("_pooled", "")
    return stem.split("_")[-1] in ("auroc", "f1max", "ap", "aupro")


def as_percentages(table, decimals=2):
    shown = table.copy()
    for column in shown.columns:
        if shown[column].dtype.kind != "f":
            continue
        shown[column] = ((100 * shown[column]).round(decimals) if _is_rate(column)
                         else shown[column].round(4))
    return shown


def clean_headline(dataset_table):
    clean = dataset_table[dataset_table["corruption"] == "clean"]
    return as_percentages(
        clean.set_index(["dataset", "prompt_mode", "model"])[HEADLINE]
        .sort_index())


def backbone_delta(dataset_table, left="tips", right="clip"):
    """`left` minus `right`, in points, on the clean setting."""
    clean = dataset_table[dataset_table["corruption"] == "clean"]
    present = set(clean["model"].unique())
    if left not in present or right not in present:
        print(f"backbone_delta skipped: need {left!r} and {right!r}, have {sorted(present)}")
        return pd.DataFrame()
    wide = clean.pivot_table(index=["dataset", "prompt_mode"], columns="model",
                             values=[m for m in HEADLINE if _is_rate(m)])
    delta = pd.DataFrame({metric: 100 * (wide[(metric, left)] - wide[(metric, right)])
                          for metric in wide.columns.levels[0]
                          if {left, right} <= set(wide[metric].columns)})
    return delta.round(2)


def all_backbone_deltas(dataset_table, reference="clip"):
    """Pairwise clean deltas of every other model against `reference`."""
    present = [name for name in MODELS
               if name in set(dataset_table["model"].unique()) and name != reference]
    frames = {}
    for name in present:
        frame = backbone_delta(dataset_table, left=name, right=reference)
        if not frame.empty:
            frames[f"{name}_minus_{reference}"] = frame
    return frames


def prompt_mode_table(dataset_table):
    """Comparable to Tipsomaly Table 2(a): fixed vs learned vs decoupled."""
    clean = dataset_table[dataset_table["corruption"] == "clean"]
    return as_percentages(
        clean.pivot_table(index=["model", "prompt_mode"], columns="dataset",
                          values=["image_auroc", "image_ap",
                                  "pixel_auroc", "pixel_aupro"]))


def robustness_summary(robustness_table, prompt_mode="decoupled"):
    subset = robustness_table[robustness_table["prompt_mode"] == prompt_mode]
    columns = [f"{metric}_drop" for metric in
               ("pixel_auroc", "pixel_aupro", "image_auroc", "image_ap")]
    return as_percentages(
        subset.groupby(["dataset", "model", "group", "severity"])[columns]
        .mean().sort_index())


def plot_severity_curves(dataset_table, prompt_mode="decoupled"):
    metrics = ["image_auroc", "image_ap", "pixel_auroc", "pixel_aupro"]
    subset = dataset_table[(dataset_table["prompt_mode"] == prompt_mode)
                           & (dataset_table["corruption"] != "clean")]
    if subset.empty:
        print("no corrupted results yet")
        return
    datasets = sorted(subset["dataset"].unique())
    figure, axes = plt.subplots(len(datasets), len(metrics),
                               figsize=(4 * len(metrics), 3.2 * len(datasets)),
                               squeeze=False)
    for row, dataset in enumerate(datasets):
        for column, metric in enumerate(metrics):
            axis = axes[row][column]
            for model in sorted(subset["model"].unique()):
                series = (subset[(subset["dataset"] == dataset)
                                 & (subset["model"] == model)]
                          .groupby("severity")[metric].mean() * 100)
                axis.plot(series.index, series.values, marker="o", label=model)
                clean = dataset_table[(dataset_table["dataset"] == dataset)
                                      & (dataset_table["model"] == model)
                                      & (dataset_table["prompt_mode"] == prompt_mode)
                                      & (dataset_table["corruption"] == "clean")][metric]
                if len(clean):
                    axis.axhline(100 * float(clean.iloc[0]), linestyle=":",
                                 linewidth=1, alpha=0.6)
            axis.set_title(f"{dataset} - {metric}")
            axis.set_xlabel("severity")
            axis.set_xticks(list(SEVERITIES))
            axis.grid(alpha=0.3)
            if column == 0:
                axis.set_ylabel("points")
                axis.legend(title=f"{prompt_mode} prompts", fontsize=8)
    figure.suptitle("Degradation under corruption (dotted line = clean reference)")
    figure.tight_layout()
    plt.show()


def plot_corruption_bars(dataset_table, metric="pixel_auroc",
                         prompt_mode="decoupled"):
    subset = dataset_table[(dataset_table["prompt_mode"] == prompt_mode)
                           & (dataset_table["corruption"] != "clean")]
    if subset.empty:
        return
    pivot = (subset.groupby(["corruption", "model"])[metric].mean().unstack() * 100)
    order = [name for group in CORRUPTION_GROUPS.values() for name in group]
    pivot = pivot.reindex([name for name in order if name in pivot.index])
    axis = pivot.plot.bar(figsize=(12, 4), width=0.8)
    axis.set_ylabel(f"{metric} (points)")
    axis.set_title(f"{metric} by corruption, averaged over severities 1-{max(SEVERITIES)}"
                   f" ({prompt_mode} prompts)")
    axis.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


if DATASET_TABLE is None or getattr(DATASET_TABLE, "empty", True):
    print("[warn] DATASET_TABLE is empty -- run the sweep before this report cell")
else:
    print("=== Clean, dataset level (points; thresholds on the score scale) ===")
    display(clean_headline(DATASET_TABLE))

    print("\n=== Backbone deltas vs CLIP, clean (points) ===")
    _deltas = all_backbone_deltas(DATASET_TABLE, reference="clip")
    if not _deltas and {"tips", "clip"} <= set(DATASET_TABLE["model"].unique()):
        display(backbone_delta(DATASET_TABLE))
    else:
        for _title, _frame in _deltas.items():
            print(f"\n--- {_title} ---")
            display(_frame)

    print("\n=== Prompt modes (cf. Tipsomaly Table 2a) ===")
    display(prompt_mode_table(DATASET_TABLE))

    print("\n=== Mean degradation by corruption group and severity ===")
    if ROBUSTNESS_TABLE is None or getattr(ROBUSTNESS_TABLE, "empty", True):
        print("[warn] ROBUSTNESS_TABLE is empty")
    else:
        display(robustness_summary(ROBUSTNESS_TABLE))

    plot_severity_curves(DATASET_TABLE)
    plot_corruption_bars(DATASET_TABLE, "pixel_auroc")
    plot_corruption_bars(DATASET_TABLE, "image_auroc")

## Cell 18 — Comparison with published results

Places these measurements next to the numbers reported in **“TIPS Over Tricks:
Simple Prompts for Effective Zero-shot Anomaly Detection”**
([arXiv:2602.03594](https://arxiv.org/abs/2602.03594), Salehi et al., ICASSP 2026),
together with the three CLIP-based baselines that paper tabulates — VAND, AnomalyCLIP
and AdaCLIP.

Because pixel metrics move with the resolution they are scored at, the clean
setting is **re-scored at `PAPER_COMPARISON_RES`** here: the stored 64×64 maps are
bilinearly upsampled onto masks loaded at 256×256. That is the same shape of
protocol the published work uses (a 37×37 map upsampled onto a 518×518 mask), so
the comparison is like-for-like in a way the sweep's 64×64 numbers are not.

### What is and is not comparable

The paper's numbers are the target, not the expectation, and three differences
matter when reading the gap:

1. **Their pipeline is more than the backbone.** Tipsomaly is TIPS *plus*
   decoupled prompts, TIPS's spatial global token, injected local evidence and a
   tuned fixed-prompt ensemble. This notebook implements those pieces but is an
   independent reimplementation, and its `decoupled` row is the one to compare;
   `fixed` and `learned` are ablations.
2. **Our CLIP row is not AnomalyCLIP.** AnomalyCLIP adds DPAM attention
   replacement, learnable visual tokens and trainable projections on top of prompt
   learning. Removing all of that is the point of this notebook, so our CLIP row
   should be expected to sit *below* the published AnomalyCLIP row. The distance
   between them is a measurement of what AnomalyCLIP's internal machinery
   contributes — which is exactly the question a backbone study should isolate.
3. **Pixel F1-max is resolution-sensitive** in a way AUROC is not, because the
   positive class is a few percent of pixels and its prevalence changes with
   resampling. Treat pixel F1-max deltas as indicative.

Everything else is aligned deliberately: same backbone (TIPS-L/14 HR), same
capacity-matched CLIP, same 518px inputs, same cross-dataset protocol (train on
MVTec test → evaluate VisA, and the reverse), same 8 context tokens, same
optimiser and schedule, same seed 111, and dataset-level scores as the unweighted
mean over categories.

The published prompt-mode ablation is reproduced too: the paper reports fixed
prompts strong at image level and near-random at pixel level, learned prompts the
reverse, and decoupled taking the better of each. Reproducing that *pattern*
validates the implementation even where absolute values differ.

In [ ]:
# =============================================================================
# CELL 18 -- Comparison with published results (arXiv:2602.03594)
# =============================================================================
PAPER = "arXiv:2602.03594"

# Table 1, industrial split: (AUROC, AP, F1-max) image / (AUROC, AUPRO, F1-max) pixel.
# VAND, AnomalyCLIP and AdaCLIP as tabulated there; "Tipsomaly" is that paper's
# own TIPS-L/14 HR pipeline.
PUBLISHED_IMAGE = {
    ("VAND", "mvtec"): (86.1, 93.5, 88.9),
    ("AnomalyCLIP", "mvtec"): (91.5, 96.2, 92.7),
    ("AdaCLIP", "mvtec"): (89.2, 95.7, 90.6),
    ("Tipsomaly", "mvtec"): (93.4, 96.1, 92.9),
    ("VAND", "visa"): (78.0, 81.4, 80.7),
    ("AnomalyCLIP", "visa"): (82.1, 85.4, 80.4),
    ("AdaCLIP", "visa"): (85.8, 79.0, 83.1),
    ("Tipsomaly", "visa"): (87.7, 90.9, 84.8),
}
PUBLISHED_PIXEL = {
    ("VAND", "mvtec"): (87.6, 44.0, 39.8),
    ("AnomalyCLIP", "mvtec"): (91.1, 81.4, 39.1),
    ("AdaCLIP", "mvtec"): (88.7, 37.8, 43.4),
    ("Tipsomaly", "mvtec"): (90.9, 84.0, 43.8),
    ("VAND", "visa"): (94.2, 86.8, 32.3),
    ("AnomalyCLIP", "visa"): (95.5, 87.0, 28.3),
    ("AdaCLIP", "visa"): (95.5, 72.9, 37.7),
    ("Tipsomaly", "visa"): (95.9, 88.2, 31.5),
}
PUBLISHED_METHODS = ("VAND", "AnomalyCLIP", "AdaCLIP", "Tipsomaly")

# Table 2(a): (image AUROC, image AP) and (pixel AUROC, pixel AUPRO) for TIPS.
PUBLISHED_PROMPT_ABLATION = {
    ("fixed", "mvtec"): (93.4, 96.1, 55.4, 24.6),
    ("fixed", "visa"): (87.7, 90.9, 43.1, 11.5),
    ("learned", "mvtec"): (84.4, 92.9, 90.9, 84.0),
    ("learned", "visa"): (67.1, 72.9, 95.9, 88.2),
    ("decoupled", "mvtec"): (93.4, 96.1, 90.9, 84.0),
    ("decoupled", "visa"): (87.7, 90.9, 95.9, 88.2),
}

# Table 8: TIPS variants, and Table 4: capacity of the two families.
PUBLISHED_TIPS_VARIANTS = {
    "TIPS-S/14 HR": (87.3, 93.5, 90.1, 89.6, 80.7, 38.6, 55.2),
    "TIPS-B/14 HR": (91.8, 96.3, 92.4, 89.0, 84.8, 39.7, 195.3),
    "TIPS-L/14 HR": (93.4, 96.1, 92.9, 90.9, 84.0, 43.8, 487.1),
    "TIPS-g/14 HR": (92.9, 96.0, 92.9, 90.6, 77.1, 43.3, 1500.0),
}

# Table 9: SigLIP2 as backbone under loss ablations (not prompt-mode ablations).
# Columns: image (AUROC, AP, F1-max) and pixel (AUROC, AUPRO, F1-max).
PUBLISHED_SIGLIP2_LOSS = {
    ("No Learning", "mvtec"): (88.7, 94.4, 90.1, 47.3, 0.1, 6.6),
    ("No Learning", "visa"): (74.9, 80.5, 78.1, 47.0, 0.0, 1.2),
    ("Local Loss", "mvtec"): (60.4, 82.1, 83.8, 61.3, 31.0, 10.3),
    ("Local Loss", "visa"): (43.2, 54.5, 72.7, 56.9, 27.9, 2.2),
    ("Global Loss", "mvtec"): (91.6, 96.6, 92.1, 48.9, 0.2, 6.8),
    ("Global Loss", "visa"): (71.4, 76.2, 76.1, 48.4, 0.0, 1.3),
    ("Both Losses", "mvtec"): (67.1, 85.4, 84.3, 57.5, 25.8, 9.6),
    ("Both Losses", "visa"): (74.4, 79.0, 77.9, 57.2, 26.3, 2.0),
}

# Explicit absences from the paper — do not invent numbers for these.
PAPER_MISSING_BACKBONES = ("tips_v2", "dinov2", "dinov3")


def clean_metrics_at(resolution=PAPER_COMPARISON_RES, verbose=True):
    """Re-scores the clean setting at `resolution`, the published protocol's shape.

    The stored low-resolution maps are upsampled onto masks loaded at the target
    resolution, exactly as published pipelines upsample a 37x37 map onto the
    full-resolution mask.
    """
    rows = []
    for model_name in MODELS:
        for mode in PROMPT_MODES:
            for _, evaluate_on in PROTOCOL:
                for category in CATEGORIES[evaluate_on]:
                    shard = load_shard(model_name, mode, evaluate_on, category,
                                       "clean", 0)
                    if shard is None:
                        continue
                    records = index_test_split(evaluate_on, category)
                    if len(records) != len(shard["labels"]):
                        continue
                    masks = np.stack([load_mask_full(record["mask"], resolution)
                                      for record in records])
                    rows.append({
                        "model": model_name, "prompt_mode": mode,
                        "dataset": evaluate_on, "category": category,
                        **evaluate(masks, resize_maps(shard["maps"], resolution),
                                   shard["labels"], shard["scores"])})
    table = pd.DataFrame(rows)
    if table.empty:
        if verbose:
            print("no clean shards found; run the sweep first")
        return table
    return (table.groupby(["model", "prompt_mode", "dataset"])[list(ALL_METRICS)]
            .mean().reset_index())


def published_frame():
    rows = []
    for method in PUBLISHED_METHODS:
        for dataset in ("mvtec", "visa"):
            image, pixel = PUBLISHED_IMAGE[(method, dataset)], PUBLISHED_PIXEL[(method, dataset)]
            rows.append({"method": f"{method} (published)", "dataset": dataset,
                         "image_auroc": image[0], "image_ap": image[1],
                         "image_f1max": image[2], "pixel_auroc": pixel[0],
                         "pixel_aupro": pixel[1], "pixel_f1max": pixel[2]})
    return pd.DataFrame(rows)


COMPARISON_COLUMNS = ["image_auroc", "image_ap", "image_f1max",
                      "pixel_auroc", "pixel_aupro", "pixel_f1max"]


def comparison_table(measured):
    """Published rows and ours, in points, side by side."""
    if measured.empty:
        return measured
    ours = measured.copy()
    ours["method"] = ("this notebook: " + ours["model"] + " / " + ours["prompt_mode"])
    for column in COMPARISON_COLUMNS:
        ours[column] = 100 * ours[column]
    combined = pd.concat([published_frame(),
                          ours[["method", "dataset"] + COMPARISON_COLUMNS]],
                         ignore_index=True)
    return (combined.pivot_table(index="method", columns="dataset",
                                 values=COMPARISON_COLUMNS)
            .round(2)
            .reindex([f"{method} (published)" for method in PUBLISHED_METHODS]
                     + sorted(ours["method"].unique())))


def gap_to_paper(measured, reference="Tipsomaly", prompt_mode="decoupled"):
    """Our decoupled rows minus the paper's, in points. Negative = below theirs."""
    if measured is None or getattr(measured, "empty", True) or "prompt_mode" not in measured:
        return pd.DataFrame()
    rows = []
    for _, row in measured[measured["prompt_mode"] == prompt_mode].iterrows():
        image = PUBLISHED_IMAGE[(reference, row["dataset"])]
        pixel = PUBLISHED_PIXEL[(reference, row["dataset"])]
        published = dict(zip(COMPARISON_COLUMNS,
                             (image[0], image[1], image[2], pixel[0], pixel[1], pixel[2])))
        rows.append({"model": row["model"], "dataset": row["dataset"],
                     **{f"{metric}_delta": round(100 * row[metric] - published[metric], 2)
                        for metric in COMPARISON_COLUMNS}})
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).set_index(["dataset", "model"]).sort_index()


def prompt_ablation_comparison(measured, model_name="tips"):
    """Our fixed/learned/decoupled rows against the paper's Table 2(a)."""
    rows = []
    subset = measured[measured["model"] == model_name]
    for _, row in subset.iterrows():
        key = (row["prompt_mode"], row["dataset"])
        published = PUBLISHED_PROMPT_ABLATION.get(key)
        rows.append({
            "dataset": row["dataset"], "prompts": row["prompt_mode"],
            "image_auroc": round(100 * row["image_auroc"], 1),
            "image_auroc_paper": published[0] if published else np.nan,
            "image_ap": round(100 * row["image_ap"], 1),
            "image_ap_paper": published[1] if published else np.nan,
            "pixel_auroc": round(100 * row["pixel_auroc"], 1),
            "pixel_auroc_paper": published[2] if published else np.nan,
            "pixel_aupro": round(100 * row["pixel_aupro"], 1),
            "pixel_aupro_paper": published[3] if published else np.nan,
        })
    order = {mode: index for index, mode in enumerate(("fixed", "learned", "decoupled"))}
    table = pd.DataFrame(rows)
    if table.empty:
        return table
    return (table.assign(_order=table["prompts"].map(order))
            .sort_values(["dataset", "_order"]).drop(columns="_order")
            .set_index(["dataset", "prompts"]))


def plot_paper_comparison(measured, metric="pixel_auroc"):
    if measured.empty:
        return
    source = PUBLISHED_PIXEL if metric.startswith("pixel") else PUBLISHED_IMAGE
    slot = {"auroc": 0, "ap": 1, "aupro": 1, "f1max": 2}[metric.split("_")[-1]]
    figure, axes = plt.subplots(1, 2, figsize=(13, 4), squeeze=False)
    for axis, dataset in zip(axes[0], ("mvtec", "visa")):
        names = [f"{method}\n(published)" for method in PUBLISHED_METHODS]
        values = [source[(method, dataset)][slot] for method in PUBLISHED_METHODS]
        subset = measured[(measured["dataset"] == dataset)
                          & (measured["prompt_mode"] == "decoupled")]
        for _, row in subset.iterrows():
            names.append(f"{row['model']}\n(ours)")
            values.append(100 * row[metric])
        colours = ["#b0b7c3"] * len(PUBLISHED_METHODS) + ["#2f6fb3"] * (len(names) - len(PUBLISHED_METHODS))
        axis.bar(names, values, color=colours)
        axis.set_title(f"{dataset} - {metric}")
        axis.set_ylabel("points")
        axis.grid(axis="y", alpha=0.3)
        for index, value in enumerate(values):
            axis.text(index, value + 0.5, f"{value:.1f}", ha="center", fontsize=8)
    figure.suptitle(f"{metric}: published results vs. this notebook "
                    f"(decoupled prompts, scored at {PAPER_COMPARISON_RES}px)")
    figure.tight_layout()
    plt.show()


MEASURED_CLEAN = clean_metrics_at(PAPER_COMPARISON_RES)

print(f"=== Published vs. measured, dataset level in points "
      f"(ours re-scored at {PAPER_COMPARISON_RES}px) ===")
if MEASURED_CLEAN is None or getattr(MEASURED_CLEAN, "empty", True):
    print("[warn] no measured clean shards yet -- showing published numbers only")
    display(published_frame().set_index(["method", "dataset"]))
else:
    display(comparison_table(MEASURED_CLEAN))
    print(f"\n=== Our decoupled rows minus Tipsomaly ({PAPER} Table 1), points ===")
    display(gap_to_paper(MEASURED_CLEAN))
    print(f"\n=== Prompt-mode ablation vs. {PAPER} Table 2(a), TIPS backbone ===")
    display(prompt_ablation_comparison(MEASURED_CLEAN, "tips"))
    plot_paper_comparison(MEASURED_CLEAN, "pixel_auroc")
    plot_paper_comparison(MEASURED_CLEAN, "image_auroc")

print("\n=== Reference: TIPS variants as published (Table 8 / Table 4) ===")
display(pd.DataFrame.from_dict(
    PUBLISHED_TIPS_VARIANTS, orient="index",
    columns=["mvtec_image_auroc", "mvtec_image_ap", "mvtec_image_f1max",
             "mvtec_pixel_auroc", "mvtec_pixel_aupro", "mvtec_pixel_f1max",
             "params_M"]))

print(f"\n=== Reference: SigLIP2 loss ablation from {PAPER} Table 9 ===")
_siglip_rows = []
for (loss, dataset), values in PUBLISHED_SIGLIP2_LOSS.items():
    _siglip_rows.append({
        "loss": loss, "dataset": dataset,
        "image_auroc": values[0], "image_ap": values[1], "image_f1max": values[2],
        "pixel_auroc": values[3], "pixel_aupro": values[4], "pixel_f1max": values[5],
    })
display(pd.DataFrame(_siglip_rows).set_index(["loss", "dataset"]).sort_index())

print(f"\nBackbones with NO published numbers in {PAPER}: "
      f"{', '.join(PAPER_MISSING_BACKBONES)}")

## Cell 19 — Qualitative examples

Reads back the stored maps and shows, for a chosen category and corruption, the
input, the ground truth, and each backbone's anomaly map side by side. The maps
come from the artefacts rather than a fresh forward pass, which doubles as a check
that what was written is what was reported.

Each map is displayed on its own colour scale: the interest is in *where* the
evidence lands, and a shared scale would hide the localisation of whichever
backbone produces lower absolute probabilities. The score printed above each map
is the stored image-level score, so a case where the map is right but the score is
wrong (or the reverse) is visible directly — which is the failure mode decoupled
prompting exists to fix.

In [ ]:
# =============================================================================
# CELL 19 -- Qualitative examples from the stored artefacts
# =============================================================================
def show_examples(dataset=None, category=None, corruption="clean", severity=0,
                  prompt_mode="decoupled", count=4, anomalous_only=True):
    dataset = dataset or PROTOCOL[0][1]
    category = category or CATEGORIES[dataset][0]
    shards = {name: load_shard(name, prompt_mode, dataset, category,
                              corruption, severity) for name in MODELS}
    shards = {name: shard for name, shard in shards.items() if shard is not None}
    if not shards:
        print(f"no artefacts for {dataset}/{category}/{corruption} s{severity}")
        return

    data = AnomalyTestSet(dataset, category, corruption, severity)
    labels = next(iter(shards.values()))["labels"]
    candidates = [index for index, label in enumerate(labels)
                  if label == 1 or not anomalous_only][:count]

    columns = 2 + len(shards)
    figure, axes = plt.subplots(len(candidates), columns,
                               figsize=(3.1 * columns, 3.1 * len(candidates)),
                               squeeze=False)
    for row, index in enumerate(candidates):
        item = data[index]
        axes[row][0].imshow(item["image"].permute(1, 2, 0).numpy())
        axes[row][0].set_title(f"{category} #{index}"
                               f"{'' if corruption == 'clean' else f' / {corruption} s{severity}'}",
                               fontsize=9)
        axes[row][1].imshow(item["mask"].numpy(), cmap="gray")
        axes[row][1].set_title("ground truth", fontsize=9)
        for column, (name, shard) in enumerate(shards.items(), start=2):
            axes[row][column].imshow(shard["maps"][index], cmap="jet")
            axes[row][column].set_title(f"{name}  score {shard['scores'][index]:.3f}",
                                        fontsize=9)
        for axis in axes[row]:
            axis.axis("off")
    figure.suptitle(f"{dataset}/{category} - {prompt_mode} prompts "
                    f"(maps at {MAP_RES}px, read back from disk)")
    figure.tight_layout()
    plt.show()


show_examples(corruption="clean", severity=0)
show_examples(corruption="gaussian_noise", severity=3)
show_examples(corruption="rotation", severity=3)

## Summary

### What was run

A backbone comparison for zero-shot anomaly detection under AnomalyCLIP's
protocol with **all internal adaptation removed**, matching the pptx Backbone
Benchmarking track. Defaults use `PPTX_MODELS`: TIPS 1, TIPS 2, CLIP, SigLIP2,
DINOv2.txt at identical 518×518 inputs, with **prompt context vectors as the only
trainable parameters** (2 × 8 × D). No DPAM, adapters, projection heads, or
learnable visual tokens. Tipsomaly publishes TIPS and SigLIP2 — not TIPS-v2 or
DINOv2.txt. DINOv3.txt remains blocked.

| | |
| --- | --- |
| Datasets | MVTec-AD and VisA, **test splits only** |
| Protocol | prompts fitted on one dataset's test split, evaluated on the other's; no shared image, no shared category |
| Prompt modes | fixed (0 trainable), learned, decoupled — all three from one forward pass |
| Corruptions | 11 perturbations in 4 groups (noise, blur, photometric, geometric) × severities 1–3, plus clean |
| Seed | 111, with per-image derived seeds so a resumed sweep reproduces an uninterrupted one |
| Stored | low-resolution anomaly maps (float16, 64×64) and image-level scores, per model × mode × category × corruption × severity |
| Metrics | pixel AUROC / F1-max / AUPRO / optimum threshold; image AUROC / F1-max / AP / optimum threshold; per category and per dataset |

### How to read the output

- **Cell 17** is the primary result: clean dataset-level metrics per backbone, the
  TIPS-minus-CLIP delta, the prompt-mode contrast, and degradation by corruption
  group and severity.
- **Cell 18** compares against the published numbers of Tipsomaly
  ([arXiv:2602.03594](https://arxiv.org/abs/2602.03594)), AnomalyCLIP, AdaCLIP and
  VAND, re-scoring the clean setting at 256px so pixel metrics are comparable in
  kind. Our CLIP row is expected to sit below the published AnomalyCLIP row —
  that distance *is* the contribution of the internal machinery this notebook
  deliberately removes.
- **`results/tables/*.csv`** carry every number at category level, keyed by a
  configuration fingerprint.

### Honest caveats

- **Pixel metrics in the sweep are computed at 64×64**, the stored map resolution,
  so every swept number is reproducible from the artefacts with no hidden
  resampling. Only the published-comparison cell re-scores at 256px. Absolute
  pixel F1-max in particular is resolution-sensitive.
- **Geometric severities are ours.** Hendrycks & Dietterich calibrate the noise,
  blur and photometric severities; the rotation / magnification / shift magnitudes
  are defined in the config cell and labelled as our choice. Their masks are warped
  with the image, so pixel metrics stay valid.
- **ImageNet-C severities were calibrated at 224px** and are applied here at 518px,
  so blur radii are relatively narrower than in the original benchmark. Identical
  for both backbones, so the comparison holds, but absolute drops are not
  comparable to ImageNet-C literature.
- **Dense-layer selection differs per backbone by design** (TIPS: last block only;
  CLIP: four aggregated blocks), following Tipsomaly's Table 7, which shows
  aggregation helps CLIP and hurts TIPS. Set `SHARED_DENSE_LAYERS` for the strictly
  matched ablation.
- **The fixed-prompt ensemble uses the real category name**; the learned prompts are
  object-agnostic. `FIXED_PROMPT_CLASS_NAME = "object"` makes both category-blind.
- **This is a reimplementation**, not the authors' code. Where our numbers differ
  from theirs, the implementation is the first suspect — which is why the
  prompt-mode ablation is reproduced: matching the published *pattern* (fixed
  strong at image level, learned strong at pixel level, decoupled taking both) is
  the evidence that the pipeline behaves as intended.